# EKF Sensor Fusion Offline Analysis

This notebook reproduces the onboard Extended Kalman Filter (error-state, quaternion attitude)
by calling the **actual C functions** from `kalman_core.c` via `ctypes`.

**Workflow:**
1. Clone repo from GitHub into Colab
2. Compile `kalman_core.c` into a shared library with `make`
3. Load via ctypes and mirror the C structs in Python
4. Read time-series CSV data (IMU @ 200Hz, UWB @ 50Hz per anchor set)
5. Step through each measurement, calling predict/update exactly as the firmware does
6. Log and plot position, velocity, attitude, covariance over time

> You can edit C files directly in Colab and rerun the build cell to tune baked-in constants.


## 1. Setup: Clone Repository & Build Shared Library

Run the next cell to clone the firmware repository into Colab.

- If the repo is public, default settings work.
- If private, set `GITHUB_TOKEN` and use HTTPS token auth.
- Set `REPO_BRANCH` if you need a non-main branch.
- Place CSV files in `analysis/data/` inside the cloned repo (or upload there in Colab).


In [ ]:
import os
import shutil
import platform

# --- Git clone configuration ---
REPO_URL = 'https://github.com/aidanquandt/hybrid-localization-system.git'
REPO_BRANCH = 'eskf-analysis'
CLONE_PARENT = '/content'
CLONE_DIRNAME = 'hybrid-localization-system'
GITHUB_TOKEN = ''  # optional: set if repo is private

clone_target = os.path.join(CLONE_PARENT, CLONE_DIRNAME)
if os.path.exists(clone_target):
    shutil.rmtree(clone_target)

if GITHUB_TOKEN.strip():
    auth_url = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')
else:
    auth_url = REPO_URL

cmd = f"git clone --depth 1 --branch {REPO_BRANCH} {auth_url} {clone_target}"
ret = os.system(cmd)
assert ret == 0, 'git clone failed. Check REPO_URL/REPO_BRANCH/token.'

REPO_ROOT = clone_target

# Locate analysis/ robustly (repo may be nested under one top-level folder)
def find_analysis_dir(root):
    direct = os.path.join(root, 'analysis')
    if os.path.isdir(direct):
        return direct

    candidates = []
    for dirpath, dirnames, _ in os.walk(root):
        # prune common heavy dirs for speed
        dirnames[:] = [d for d in dirnames if d not in {'.git', '.venv', '__pycache__', 'build', 'dist', 'node_modules'}]
        if os.path.basename(dirpath) == 'analysis':
            candidates.append(dirpath)

    if not candidates:
        return None

    # Prefer shallowest path under clone root
    candidates.sort(key=lambda p: p.count(os.sep))
    return candidates[0]

ANALYSIS_DIR = find_analysis_dir(REPO_ROOT)
assert ANALYSIS_DIR is not None, f'analysis/ not found anywhere under {REPO_ROOT}'

DATA_DIR = os.path.join(ANALYSIS_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

print(f'Repo root:    {REPO_ROOT}')
print(f'Analysis dir: {ANALYSIS_DIR}')
print(f'Data dir:     {DATA_DIR}')
print(f'Platform:     {platform.system()} {platform.machine()}')
print('Tip: edit C files in Colab and rerun build cell to apply baked-in parameter changes.')


In [ ]:
# Build the shared library from the cloned repo
os.chdir(ANALYSIS_DIR)
ret = os.system('make clean && make')
assert ret == 0, 'Build failed: check compiler output above.'

# Determine library path
if platform.system() == 'Darwin':
    LIB_PATH = os.path.join(ANALYSIS_DIR, 'libkalman.dylib')
else:
    LIB_PATH = os.path.join(ANALYSIS_DIR, 'libkalman.so')

assert os.path.exists(LIB_PATH), f'Library not found at {LIB_PATH}'
print(f'Library built: {LIB_PATH}')
print('If you change kalman_core.c or headers, rerun this cell before analysis.')


## 2. ctypes Bindings

In [ ]:
import ctypes
from ctypes import c_float, c_uint8, c_uint16, c_uint32, c_bool, POINTER, Structure, byref
import numpy as np

# --- Constants (must match kalman_core.h) ---
KC_STATE_DIM = 9

# State indices
KC_STATE_X  = 0  # Position X (world, m)
KC_STATE_Y  = 1  # Position Y (world, m)
KC_STATE_Z  = 2  # Position Z (world, m)
KC_STATE_PX = 3  # Velocity X (body, m/s)
KC_STATE_PY = 4  # Velocity Y (body, m/s)
KC_STATE_PZ = 5  # Velocity Z (body, m/s)
KC_STATE_D0 = 6  # Attitude error roll (rad)
KC_STATE_D1 = 7  # Attitude error pitch (rad)
KC_STATE_D2 = 8  # Attitude error yaw (rad)


# --- C struct mirrors ---

class ArmMatrixInstanceF32(Structure):
    """Mirrors arm_matrix_instance_f32"""
    _fields_ = [
        ('numRows', c_uint16),
        ('numCols', c_uint16),
        ('pData', POINTER(c_float)),
    ]


class Axis3f(Structure):
    """Mirrors Axis3f (3-axis float vector)"""
    _fields_ = [
        ('x', c_float),
        ('y', c_float),
        ('z', c_float),
    ]


class DistanceMeasurement(Structure):
    """Mirrors distanceMeasurement_t"""
    _fields_ = [
        ('x', c_float),
        ('y', c_float),
        ('z', c_float),
        ('distance', c_float),
        ('stdDev', c_float),
        ('anchorId', c_uint8),
    ]


class KalmanCoreParams(Structure):
    """Mirrors kalmanCoreParams_t"""
    _fields_ = [
        ('stdDevInitialPosition_xy', c_float),
        ('stdDevInitialPosition_z', c_float),
        ('stdDevInitialVelocity', c_float),
        ('stdDevInitialAttitude_rollpitch', c_float),
        ('stdDevInitialAttitude_yaw', c_float),
        ('procNoiseAcc_xy', c_float),
        ('procNoiseAcc_z', c_float),
        ('procNoiseVel', c_float),
        ('procNoisePos', c_float),
        ('procNoiseAtt', c_float),
        ('measNoiseGyro_rollpitch', c_float),
        ('measNoiseGyro_yaw', c_float),
        ('initialX', c_float),
        ('initialY', c_float),
        ('initialZ', c_float),
        ('initialYaw', c_float),
    ]


# Covariance matrix type: float[9][9]
CovMatrix = (c_float * KC_STATE_DIM) * KC_STATE_DIM
# Rotation matrix type: float[3][3]
RotMatrix = (c_float * 3) * 3


class KalmanCoreData(Structure):
    """Mirrors kalmanCoreData_t"""
    _fields_ = [
        ('S', c_float * KC_STATE_DIM),         # State vector
        ('q', c_float * 4),                     # Quaternion [w, x, y, z]
        ('R', RotMatrix),                       # Rotation matrix (body to world)
        ('P', CovMatrix),                       # Covariance matrix (9x9)
        ('Pm', ArmMatrixInstanceF32),           # ARM matrix instance for P
        ('initialQuaternion', c_float * 4),     # Initial quaternion
        ('isUpdated', c_bool),                  # Update flag
        ('lastPredictionMs', c_uint32),         # Last prediction timestamp
        ('lastProcessNoiseUpdateMs', c_uint32), # Last process noise timestamp
    ]


print(f'KalmanCoreData size: {ctypes.sizeof(KalmanCoreData)} bytes')
print(f'KalmanCoreParams size: {ctypes.sizeof(KalmanCoreParams)} bytes')

In [ ]:
# --- Load shared library and declare function signatures ---

lib = ctypes.CDLL(LIB_PATH)

# void kalmanCoreDefaultParams(kalmanCoreParams_t* params)
lib.kalmanCoreDefaultParams.argtypes = [POINTER(KalmanCoreParams)]
lib.kalmanCoreDefaultParams.restype = None

# void kalmanCoreInit(kalmanCoreData_t* kf, const kalmanCoreParams_t* params, uint32_t nowMs)
lib.kalmanCoreInit.argtypes = [POINTER(KalmanCoreData), POINTER(KalmanCoreParams), c_uint32]
lib.kalmanCoreInit.restype = None

# void kalmanCorePredict(kalmanCoreData_t* kf, const kalmanCoreParams_t* params,
#                        Axis3f* acc, Axis3f* gyro, uint32_t nowMs)
lib.kalmanCorePredict.argtypes = [
    POINTER(KalmanCoreData), POINTER(KalmanCoreParams),
    POINTER(Axis3f), POINTER(Axis3f), c_uint32
]
lib.kalmanCorePredict.restype = None

# void kalmanCoreAddProcessNoise(kalmanCoreData_t* kf, const kalmanCoreParams_t* params,
#                                uint32_t nowMs)
lib.kalmanCoreAddProcessNoise.argtypes = [
    POINTER(KalmanCoreData), POINTER(KalmanCoreParams), c_uint32
]
lib.kalmanCoreAddProcessNoise.restype = None

# void kalmanCoreUpdateWithDistance(kalmanCoreData_t* kf, distanceMeasurement_t* d)
lib.kalmanCoreUpdateWithDistance.argtypes = [
    POINTER(KalmanCoreData), POINTER(DistanceMeasurement)
]
lib.kalmanCoreUpdateWithDistance.restype = None

# bool kalmanCoreFinalize(kalmanCoreData_t* kf)
lib.kalmanCoreFinalize.argtypes = [POINTER(KalmanCoreData)]
lib.kalmanCoreFinalize.restype = c_bool

# void kalmanCoreGetPosition(const kalmanCoreData_t* kf, float* x, float* y, float* z)
lib.kalmanCoreGetPosition.argtypes = [
    POINTER(KalmanCoreData), POINTER(c_float), POINTER(c_float), POINTER(c_float)
]
lib.kalmanCoreGetPosition.restype = None

# void kalmanCoreGetVelocity(const kalmanCoreData_t* kf, float* vx, float* vy, float* vz)
lib.kalmanCoreGetVelocity.argtypes = [
    POINTER(KalmanCoreData), POINTER(c_float), POINTER(c_float), POINTER(c_float)
]
lib.kalmanCoreGetVelocity.restype = None

# void kalmanCoreGetAttitude(const kalmanCoreData_t* kf, float* roll, float* pitch, float* yaw)
lib.kalmanCoreGetAttitude.argtypes = [
    POINTER(KalmanCoreData), POINTER(c_float), POINTER(c_float), POINTER(c_float)
]
lib.kalmanCoreGetAttitude.restype = None

# void kalmanCoreGetQuaternion(const kalmanCoreData_t* kf,
#                              float* qw, float* qx, float* qy, float* qz)
lib.kalmanCoreGetQuaternion.argtypes = [
    POINTER(KalmanCoreData),
    POINTER(c_float), POINTER(c_float), POINTER(c_float), POINTER(c_float)
]
lib.kalmanCoreGetQuaternion.restype = None

print('Library loaded and function signatures declared.')

## 3. Helper Functions

In [ ]:
def init_filter(params=None, init_timestamp_ms=0):
    """
    Initialize the Kalman filter with default or custom parameters.
    Returns (kf_data, kf_params) ctypes structs.
    """
    kf_params = KalmanCoreParams()
    lib.kalmanCoreDefaultParams(byref(kf_params))

    # Override with custom params if provided
    if params is not None:
        for field_name, _ in KalmanCoreParams._fields_:
            if field_name in params:
                setattr(kf_params, field_name, params[field_name])

    kf_data = KalmanCoreData()
    lib.kalmanCoreInit(byref(kf_data), byref(kf_params), c_uint32(int(init_timestamp_ms)))

    return kf_data, kf_params


def get_state(kf_data):
    """Extract full state from filter as a dict."""
    x, y, z = c_float(), c_float(), c_float()
    vx, vy, vz = c_float(), c_float(), c_float()
    roll, pitch, yaw = c_float(), c_float(), c_float()
    qw, qx, qy, qz = c_float(), c_float(), c_float(), c_float()

    lib.kalmanCoreGetPosition(byref(kf_data), byref(x), byref(y), byref(z))
    lib.kalmanCoreGetVelocity(byref(kf_data), byref(vx), byref(vy), byref(vz))
    lib.kalmanCoreGetAttitude(byref(kf_data), byref(roll), byref(pitch), byref(yaw))
    lib.kalmanCoreGetQuaternion(byref(kf_data), byref(qw), byref(qx), byref(qy), byref(qz))

    # Extract covariance diagonal
    P_diag = [kf_data.P[i][i] for i in range(KC_STATE_DIM)]

    return {
        'x': x.value, 'y': y.value, 'z': z.value,
        'vx': vx.value, 'vy': vy.value, 'vz': vz.value,
        'roll': roll.value, 'pitch': pitch.value, 'yaw': yaw.value,
        'qw': qw.value, 'qx': qx.value, 'qy': qy.value, 'qz': qz.value,
        'P_diag': P_diag,
    }


def process_imu(kf_data, kf_params, timestamp_ms, ax, ay, az, gx, gy, gz):
    """Process one IMU measurement: predict + process noise + finalize."""
    acc = Axis3f(ax, ay, az)
    gyro = Axis3f(gx, gy, gz)
    ts = c_uint32(int(timestamp_ms))

    lib.kalmanCorePredict(byref(kf_data), byref(kf_params), byref(acc), byref(gyro), ts)
    lib.kalmanCoreAddProcessNoise(byref(kf_data), byref(kf_params), ts)
    lib.kalmanCoreFinalize(byref(kf_data))


def process_uwb(kf_data, anchor_x, anchor_y, anchor_z, distance, stddev, anchor_id):
    """Process one UWB range measurement: update + finalize."""
    d = DistanceMeasurement(
        x=anchor_x, y=anchor_y, z=anchor_z,
        distance=distance, stdDev=stddev, anchorId=int(anchor_id)
    )
    lib.kalmanCoreUpdateWithDistance(byref(kf_data), byref(d))
    lib.kalmanCoreFinalize(byref(kf_data))


print('Helper functions defined.')


## 4. Load CSV Data

Current functional event format uses mixed event rows.

Required columns:
- `timestamp_ms`: timestamp in milliseconds.
- `type`: event type (`IMU`, `RANGING`, or `POSITION`).

IMU columns:
- `accel_x`, `accel_y`, `accel_z` (m/s^2)
- `gyro_x`, `gyro_y`, `gyro_z` (rad/s)

RANGING columns:
- `dist_m` (preferred) or legacy `distance`
- `anchor_addr` (preferred) or legacy `anchor_id`
- `anchor_x`, `anchor_y`, `anchor_z`
- optional `stddev` (defaults to `DEFAULT_UWB_STDDEV` if missing)

POSITION columns (reference only):
- `pos_x`, `pos_y`, `pos_z`
- `vel_x`, `vel_y`, `vel_z`
- `confidence`

Offline fusion behavior in this notebook:
- `IMU` rows always run EKF prediction when valid IMU fields are present.
- Every `RANGING` row runs EKF range update when required fields are valid.
- `POSITION` rows are not fused (reference/visualization + IMU enable timeline source).

Place your CSV file in `analysis/data/` and set the filename below.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

# --- Set your CSV filename here ---
CSV_FILENAME = 'staticgriddata.CSV'  # <-- change filename here
CSV_PATH = os.path.join(DATA_DIR, CSV_FILENAME)

if not os.path.exists(CSV_PATH):
    print(f'WARNING: CSV file not found at {CSV_PATH}')
    print('Place your data file in analysis/data/ and update CSV_FILENAME above.')
    print('Skipping data load - you can still use the helper functions manually.')
    df = None
else:
    df = pd.read_csv(CSV_PATH)
    if 'timestamp_ms' in df.columns:
        df = df.sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

    print(f'Loaded {len(df)} rows from {CSV_FILENAME}')
    print(f'Columns: {list(df.columns)}')

    if len(df) > 0 and 'timestamp_ms' in df.columns:
        print(f'Time range: {df["timestamp_ms"].min()} - {df["timestamp_ms"].max()} ms')

    event_counts = df['type'].astype(str).str.upper().value_counts() if 'type' in df.columns else pd.Series(dtype=int)
    print(f'IMU samples: {int(event_counts.get("IMU", 0))}')
    print(f'RANGING samples: {int(event_counts.get("RANGING", 0))}')
    print(f'POSITION samples: {int(event_counts.get("POSITION", 0))}')
    print(f'Legacy UWB samples: {int(event_counts.get("UWB", 0))}')
    display(df.head(10))

    # Quick visualization of fused POSITION events directly from the CSV
    pos_df = df[df['type'].astype(str).str.upper() == 'POSITION'].copy() if 'type' in df.columns else pd.DataFrame()
    if len(pos_df) > 0:
        pos_df = pos_df.dropna(subset=['timestamp_ms', 'pos_x', 'pos_y']).sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

    # Ground truth rectangle: width=1.621 (x), height=1.835 (y), bottom-left shifted by (0.375, 1.553)
    gt_bl_x, gt_bl_y = 0.375, 1.553
    gt_w, gt_h = 1.621, 1.835
    gt_x = np.array([gt_bl_x, gt_bl_x + gt_w, gt_bl_x + gt_w, gt_bl_x, gt_bl_x])
    gt_y = np.array([gt_bl_y, gt_bl_y, gt_bl_y + gt_h, gt_bl_y + gt_h, gt_bl_y])

    # Anchor locations
    anchors = {
        'A4': (0.167, 4.237),
        'A3': (2.561, 4.254),
        'A6': (2.316, 0.000),
        'A8': (0.000, 0.000),
    }
    anchor_x = np.array([v[0] for v in anchors.values()])
    anchor_y = np.array([v[1] for v in anchors.values()])

    if len(pos_df) > 0:
        t_sec = (pos_df['timestamp_ms'].to_numpy() - pos_df['timestamp_ms'].iloc[0]) / 1000.0
        x = pos_df['pos_x'].to_numpy()
        y = pos_df['pos_y'].to_numpy()

        fig, ax = plt.subplots(1, 1, figsize=(7, 7))

        if len(pos_df) > 1:
            points = np.array([x, y]).T.reshape(-1, 1, 2)
            segments = np.concatenate([points[:-1], points[1:]], axis=1)
            norm = Normalize(vmin=t_sec.min(), vmax=t_sec.max())
            lc = LineCollection(segments, cmap='turbo', norm=norm)
            lc.set_array(t_sec[:-1])
            lc.set_linewidth(2.0)
            line = ax.add_collection(lc)
            cbar = fig.colorbar(line, ax=ax)
            cbar.set_label('Time since first plotted POSITION sample (s)')
            ax.plot(x[0], y[0], 'go', markersize=8, label='Start')
            ax.plot(x[-1], y[-1], 'rs', markersize=8, label='End')
        else:
            ax.plot(x[0], y[0], 'bo', markersize=8, label='Only POSITION sample')

        # Overlay ground truth path
        ax.plot(gt_x, gt_y, 'k--', linewidth=2.0, label='Ground truth path')

        # Overlay anchors
        ax.scatter(anchor_x, anchor_y, marker='^', s=70, c='magenta', edgecolors='black', label='Anchors')
        for name, (ax_x, ax_y) in anchors.items():
            ax.text(ax_x + 0.03, ax_y + 0.03, name, fontsize=9, color='black')

        ax.set_xlabel('pos_x (m)')
        ax.set_ylabel('pos_y (m)')
        ax.set_title('CSV POSITION 2D Trajectory + Ground Truth + Anchors (full run)')
        ax.grid(True, alpha=0.3)

        # Make axes square with equal x/y scale and square frame, covering path + GT + anchors
        all_x = np.concatenate([x, gt_x, anchor_x])
        all_y = np.concatenate([y, gt_y, anchor_y])
        x_min, x_max = np.min(all_x), np.max(all_x)
        y_min, y_max = np.min(all_y), np.max(all_y)
        cx, cy = 0.5 * (x_min + x_max), 0.5 * (y_min + y_max)
        span = max(x_max - x_min, y_max - y_min)
        if span == 0:
            span = 1.0
        pad = 0.08 * span
        half = 0.5 * span + pad
        ax.set_xlim(-2.0, 15.0)
        ax.set_ylim(-1.0, 6.0)
        ax.set_aspect('equal', adjustable='box')

        ax.legend(loc='best')
        plt.tight_layout()
        plt.show()
    else:
        print('No POSITION rows with valid timestamp_ms/pos_x/pos_y.')


In [ ]:
# Playback of POSITION trajectory at 10x speed with 100-second path history
if df is None:
    print('No data loaded. Run the CSV load cell first.')
else:
    from IPython.display import HTML, display

    pos_rt = df[df['type'].astype(str).str.upper() == 'POSITION'].copy() if 'type' in df.columns else pd.DataFrame()
    pos_rt = pos_rt.dropna(subset=['timestamp_ms', 'pos_x', 'pos_y']).sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

    if len(pos_rt) < 2:
        print('Need at least 2 POSITION samples for playback.')
    else:
        t_ms = pos_rt['timestamp_ms'].to_numpy(dtype=float)
        x = pos_rt['pos_x'].to_numpy(dtype=float)
        y = pos_rt['pos_y'].to_numpy(dtype=float)
        t_sec = (t_ms - t_ms[0]) / 1000.0

        # Playback configuration
        HISTORY_SEC = 100.0
        PLAYBACK_SPEED = 10.0  # 10x real time
        TARGET_FPS = 20.0
        MAX_RENDER_FRAMES = 1200

        # Build frame indices to keep payload manageable in Colab
        desired_data_dt = PLAYBACK_SPEED / TARGET_FPS  # seconds of data advanced per rendered frame
        dt_data = np.diff(t_sec)
        median_dt = np.median(dt_data) if len(dt_data) > 0 else desired_data_dt
        stride = max(1, int(np.round(desired_data_dt / max(median_dt, 1e-6))))
        frame_idx = np.arange(0, len(t_sec), stride, dtype=int)
        if frame_idx[-1] != len(t_sec) - 1:
            frame_idx = np.append(frame_idx, len(t_sec) - 1)

        if len(frame_idx) > MAX_RENDER_FRAMES:
            stride2 = int(np.ceil(len(frame_idx) / MAX_RENDER_FRAMES))
            frame_idx = frame_idx[::stride2]
            if frame_idx[-1] != len(t_sec) - 1:
                frame_idx = np.append(frame_idx, len(t_sec) - 1)

        # Square axis limits derived from full plotted trajectory
        x_min, x_max = np.min(x), np.max(x)
        y_min, y_max = np.min(y), np.max(y)
        cx, cy = 0.5 * (x_min + x_max), 0.5 * (y_min + y_max)
        span = max(x_max - x_min, y_max - y_min)
        if span == 0:
            span = 1.0
        pad = 0.05 * span
        half = 0.5 * span + pad

        fig, ax = plt.subplots(1, 1, figsize=(7, 7))
        ax.set_xlim(-2.0, 15.0)
        ax.set_ylim(-1.0, 6.0)
        ax.set_aspect('equal', adjustable='box')
        ax.set_xlabel('pos_x (m)')
        ax.set_ylabel('pos_y (m)')
        ax.set_title('POSITION playback (10x speed, 100 s trail, full run)')
        ax.grid(True, alpha=0.3)

        norm = Normalize(vmin=0.0, vmax=HISTORY_SEC)
        cmap = plt.get_cmap('turbo')

        lc = LineCollection([], cmap=cmap, norm=norm, linewidths=2.5)
        lc.set_array(np.array([], dtype=float))
        ax.add_collection(lc)

        current_pt, = ax.plot([], [], 'ko', markersize=5, label='Current')
        ax.plot([x[0]], [y[0]], 'go', markersize=8, label='Start')
        time_text = ax.text(0.02, 0.98, '', transform=ax.transAxes, va='top')

        cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax)
        cbar.set_label('Age within trail (s): 100 (oldest) -> 0 (newest)')
        ax.legend(loc='lower right')

        interval_ms = int(1000.0 / TARGET_FPS)

        def init():
            lc.set_segments([])
            lc.set_array(np.array([], dtype=float))
            current_pt.set_data([], [])
            time_text.set_text('t = 0.00 s')
            return lc, current_pt, time_text

        def update(frame_no):
            i = frame_idx[frame_no]
            t_now = t_sec[i]
            t_start = max(0.0, t_now - HISTORY_SEC)

            j0 = np.searchsorted(t_sec, t_start, side='left')
            xh = x[j0:i+1]
            yh = y[j0:i+1]
            th = t_sec[j0:i+1]

            if len(xh) >= 2:
                pts = np.array([xh, yh]).T.reshape(-1, 1, 2)
                segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
                age = t_now - th[:-1]
                lc.set_segments(segs)
                lc.set_array(age)
            else:
                lc.set_segments([])
                lc.set_array(np.array([], dtype=float))

            current_pt.set_data([x[i]], [y[i]])
            time_text.set_text(f't = {t_now:0.2f} s  |  trail = {min(HISTORY_SEC, t_now):0.2f} s  |  speed = {PLAYBACK_SPEED:.0f}x')
            return lc, current_pt, time_text

        # Raise embed limit so Colab doesn't truncate generated animation
        plt.rcParams['animation.embed_limit'] = 80

        anim = animation.FuncAnimation(
            fig,
            update,
            frames=len(frame_idx),
            init_func=init,
            interval=interval_ms,
            blit=False,
            repeat=False,
        )

        # Keep reference + render inline
        globals()['_position_anim_ref'] = anim
        display(HTML(anim.to_jshtml(default_mode='once')))
        plt.close(fig)


In [ ]:
# Export video: POSITION XY from 2609-2635 s with ground truth (forward playback)
if df is None:
    print('No data loaded. Run the CSV load cell first.')
else:
    import os
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.animation as animation
    from matplotlib.colors import Normalize
    from matplotlib.patches import FancyBboxPatch

    WINDOW_START_SEC = 2609.0
    WINDOW_END_SEC = 2635.0
    FPS = 20

    pos_vid = df[df['type'].astype(str).str.upper() == 'POSITION'].copy() if 'type' in df.columns else pd.DataFrame()
    pos_vid = pos_vid.dropna(subset=['timestamp_ms', 'pos_x', 'pos_y']).sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

    if len(pos_vid) < 2:
        print('Need at least 2 POSITION samples to make video.')
    else:
        # Prefer global timeline: seconds since first CSV timestamp
        global_t = (pos_vid['timestamp_ms'].to_numpy(dtype=float) - float(df['timestamp_ms'].min())) / 1000.0
        mask = (global_t >= WINDOW_START_SEC) & (global_t <= WINDOW_END_SEC)

        # Fallback to POSITION-relative timeline if global window has no data
        if np.count_nonzero(mask) < 2:
            pos_t = (pos_vid['timestamp_ms'].to_numpy(dtype=float) - float(pos_vid['timestamp_ms'].iloc[0])) / 1000.0
            mask = (pos_t >= WINDOW_START_SEC) & (pos_t <= WINDOW_END_SEC)
            if np.count_nonzero(mask) >= 2:
                print('Using POSITION-relative timeline for 2609-2635 s window (global window had insufficient data).')

        pos_vid = pos_vid[mask].reset_index(drop=True)

        if len(pos_vid) < 2:
            print(f'Not enough POSITION samples in {WINDOW_START_SEC:.1f}-{WINDOW_END_SEC:.1f} s window.')
        else:
            # Forward playback (time increasing)
            t = (pos_vid['timestamp_ms'].to_numpy(dtype=float) - pos_vid['timestamp_ms'].iloc[0]) / 1000.0
            x = pos_vid['pos_x'].to_numpy(dtype=float)
            y = pos_vid['pos_y'].to_numpy(dtype=float)

            # Transformed view with reflection about vertical axis in video frame:
            # start from (u, v) = (-y, x), then reflect: u <- -u => (u, v) = (y, x)
            x_plot_raw = y
            y_plot_raw = x

            x_plot = x_plot_raw
            y_plot = y_plot_raw

            # Ground truth rectangle (shifted down by 0.7 m), transformed the same way
            gt_bl_x, gt_bl_y = 0.375, (1.553 - 0.7)
            gt_w, gt_h = 1.621, 1.835
            gt_plot_x0 = gt_bl_y
            gt_plot_y0 = gt_bl_x
            gt_plot_w = gt_h
            gt_plot_h = gt_w

            fig, ax = plt.subplots(1, 1, figsize=(8, 8))

            # Fixed overlay: rounded-corner ground truth rectangle
            rounding = 0.10 * min(gt_plot_w, gt_plot_h)
            gt_patch = FancyBboxPatch(
                (gt_plot_x0, gt_plot_y0),
                gt_plot_w,
                gt_plot_h,
                boxstyle=f'round,pad=0,rounding_size={rounding}',
                fill=False,
                edgecolor='black',
                linestyle='--',
                linewidth=2.5,
                zorder=2,
            )
            ax.add_patch(gt_patch)

            # Dynamic trajectory (persistent forward-growing trail)
            path_line, = ax.plot([], [], color='black', linewidth=1.8, alpha=0.6, zorder=3)
            progress = np.linspace(0.0, 1.0, len(x_plot))
            norm = Normalize(vmin=0.0, vmax=1.0)
            trail_scatter = ax.scatter([], [], c=[], cmap='turbo', norm=norm, s=22, zorder=4)
            current_pt, = ax.plot([], [], 'ko', markersize=7, zorder=5)

            # Square axes and styling for inset use
            gt_outline_x = np.array([gt_plot_x0, gt_plot_x0 + gt_plot_w])
            gt_outline_y = np.array([gt_plot_y0, gt_plot_y0 + gt_plot_h])
            all_x = np.concatenate([x_plot, gt_outline_x])
            all_y = np.concatenate([y_plot, gt_outline_y])
            x_min, x_max = np.min(all_x), np.max(all_x)
            y_min, y_max = np.min(all_y), np.max(all_y)
            cx, cy = 0.5 * (x_min + x_max), 0.5 * (y_min + y_max)
            span = max(x_max - x_min, y_max - y_min)
            if span == 0:
                span = 1.0
            pad = 0.03 * span
            half = 0.5 * span + pad

            ax.set_xlim(-2.0, 15.0)
            ax.set_ylim(-1.0, 6.0)
            ax.set_aspect('equal', adjustable='box')
            ax.margins(0)

            # No axis tick values; keep axis labels only
            ax.set_xticks([])
            ax.set_yticks([])
            ax.tick_params(length=0, labelbottom=False, labelleft=False)
            ax.grid(False)
            fig.subplots_adjust(left=0, right=1, bottom=0, top=1)

            def init():
                path_line.set_data([], [])
                trail_scatter.set_offsets(np.empty((0, 2)))
                trail_scatter.set_array(np.array([], dtype=float))
                current_pt.set_data([], [])
                return path_line, trail_scatter, current_pt

            def update(i):
                xi = x_plot[:i+1]
                yi = y_plot[:i+1]
                pi = progress[:i+1]

                path_line.set_data(xi, yi)
                trail_scatter.set_offsets(np.column_stack((xi, yi)))
                trail_scatter.set_array(pi)
                current_pt.set_data([x_plot[i]], [y_plot[i]])
                return path_line, trail_scatter, current_pt

            anim = animation.FuncAnimation(
                fig,
                update,
                frames=len(x_plot),
                init_func=init,
                interval=int(1000 / FPS),
                blit=False,
                repeat=False,
            )

            globals()['_export_anim_ref'] = anim

            out_dir = DATA_DIR if 'DATA_DIR' in globals() else os.getcwd()
            mp4_path = os.path.join(out_dir, 'position_estimate_2609_2635_xy_forward_reflectY.mp4')
            saved_path = None

            try:
                writer = animation.FFMpegWriter(fps=FPS, bitrate=3000, codec='libx264')
                anim.save(mp4_path, writer=writer, dpi=180)
                saved_path = mp4_path
                print(f'MP4 saved: {saved_path}')
            except Exception as e:
                print(f'FFmpeg MP4 export failed: {e}')
                gif_path = os.path.join(out_dir, 'position_estimate_2609_2635_xy_forward_reflectY.gif')
                writer = animation.PillowWriter(fps=FPS)
                anim.save(gif_path, writer=writer, dpi=140)
                saved_path = gif_path
                print(f'GIF saved instead: {saved_path}')

            plt.close(fig)

            # Colab download helper
            try:
                from google.colab import files
                files.download(saved_path)
            except Exception:
                print(f'Download manually from: {saved_path}')


## 5. Run the Filter

In [ ]:
# Default UWB/RANGING measurement std dev (matches firmware RANGING_DEFAULT_STDDEV_M)
DEFAULT_UWB_STDDEV = 0.2


def _get_numeric(row, candidates):
    """Return first finite numeric value found in candidate columns, else None."""
    for col in candidates:
        if col in row.index and pd.notna(row[col]):
            try:
                return float(row[col])
            except (TypeError, ValueError):
                continue
    return None


def _get_int(row, candidates):
    """Return first valid int found in candidate columns, else None."""
    for col in candidates:
        if col in row.index and pd.notna(row[col]):
            try:
                return int(float(row[col]))
            except (TypeError, ValueError):
                continue
    return None


def _to_boolish(v, default=False):
    """Parse common bool-ish scalar values."""
    if pd.isna(v):
        return default
    s = str(v).strip().lower()
    if s in {'1', 'true', 't', 'yes', 'y', 'on'}:
        return True
    if s in {'0', 'false', 'f', 'no', 'n', 'off'}:
        return False
    return default


def _build_imu_enable_lookup(df):
    """
    Build piecewise-constant IMU enable timeline from POSITION rows.
    Returns (timestamps_ms, values_bool) or (None, None) if unavailable.
    """
    required = {'type', 'timestamp_ms', 'imu_enable'}
    if not required.issubset(df.columns):
        return None, None

    pos = df[df['type'].astype(str).str.upper() == 'POSITION'][['timestamp_ms', 'imu_enable']].copy()
    if len(pos) == 0:
        return None, None

    pos['timestamp_ms'] = pd.to_numeric(pos['timestamp_ms'], errors='coerce')
    pos = pos.dropna(subset=['timestamp_ms']).sort_values('timestamp_ms', kind='stable')
    if len(pos) == 0:
        return None, None

    vals = pos['imu_enable'].apply(lambda x: _to_boolish(x, default=False)).to_numpy(dtype=bool)
    ts = pos['timestamp_ms'].to_numpy(dtype=np.int64)

    # Deduplicate repeated timestamps by keeping the latest value.
    keep = np.r_[ts[1:] != ts[:-1], True]
    return ts[keep], vals[keep]


def run_filter(
    df,
    custom_params=None,
    log_interval=1,
    start_time_ms=None,
    end_time_ms=None,
    respect_imu_enable=True,
    default_imu_enabled=True,
):
    """
    Run the EKF over the dataset.

    Processing behavior:
      - IMU      -> predict + process noise
      - RANGING  -> range update
      - POSITION -> reference only (ignored by filter loop)

    Notes:
      - IMU enable gating is intentionally disabled in this notebook so IMU rows
        are always fused when valid.
    """
    if df is None or len(df) == 0:
        print('Input DataFrame is empty.')
        return pd.DataFrame()

    if 'timestamp_ms' not in df.columns or 'type' not in df.columns:
        raise ValueError("CSV must contain at least 'timestamp_ms' and 'type' columns.")

    ordered_df = df.sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

    if start_time_ms is None:
        t0 = float(pd.to_numeric(ordered_df['timestamp_ms'], errors='coerce').min())
        start_time_ms = int(t0 + 400000.0)  # start offline analysis after 400 s

    if start_time_ms is not None:
        ordered_df = ordered_df[pd.to_numeric(ordered_df['timestamp_ms'], errors='coerce') >= float(start_time_ms)]
    if end_time_ms is not None:
        ordered_df = ordered_df[pd.to_numeric(ordered_df['timestamp_ms'], errors='coerce') <= float(end_time_ms)]
    ordered_df = ordered_df.dropna(subset=['timestamp_ms']).reset_index(drop=True)

    if len(ordered_df) == 0:
        print('No rows in selected time horizon after filtering.')
        return pd.DataFrame()

    init_ts = int(float(ordered_df['timestamp_ms'].iloc[0]))
    kf_data, kf_params = init_filter(custom_params, init_timestamp_ms=init_ts)

    # Kept for output compatibility; always True now.
    imu_enabled_current = True

    log = {
        'timestamp_ms': [], 'type': [], 'imu_enabled': [],
        'x': [], 'y': [], 'z': [],
        'vx': [], 'vy': [], 'vz': [],
        'roll': [], 'pitch': [], 'yaw': [],
        'qw': [], 'qx': [], 'qy': [], 'qz': [],
    }
    state_names = ['X', 'Y', 'Z', 'PX', 'PY', 'PZ', 'D0', 'D1', 'D2']
    for name in state_names:
        log[f'P_{name}'] = []

    n_rows = len(ordered_df)
    fused_count = 0
    skipped_counts = {
        'POSITION': 0,
        'UNKNOWN': 0,
        'INVALID_RANGING': 0,
        'INVALID_IMU': 0,
        'IMU_DISABLED': 0,
    }

    for idx, row in ordered_df.iterrows():
        ts = int(float(row['timestamp_ms']))
        mtype = str(row['type']).strip().upper()
        did_fuse = False

        if mtype == 'IMU':
            ax = _get_numeric(row, ['accel_x'])
            ay = _get_numeric(row, ['accel_y'])
            az = _get_numeric(row, ['accel_z'])
            gx = _get_numeric(row, ['gyro_x'])
            gy = _get_numeric(row, ['gyro_y'])
            gz = _get_numeric(row, ['gyro_z'])

            if None in (ax, ay, az, gx, gy, gz):
                skipped_counts['INVALID_IMU'] += 1
                continue

            process_imu(kf_data, kf_params, ts, ax, ay, az, gx, gy, gz)
            did_fuse = True

        elif mtype in ('RANGING', 'UWB'):
            dist = _get_numeric(row, ['dist_m', 'distance'])
            anchor_x = _get_numeric(row, ['anchor_x'])
            anchor_y = _get_numeric(row, ['anchor_y'])
            anchor_z = _get_numeric(row, ['anchor_z'])
            anchor_id = _get_int(row, ['anchor_addr', 'anchor_id'])
            stddev = _get_numeric(row, ['stddev'])
            if stddev is None:
                stddev = DEFAULT_UWB_STDDEV

            if None in (dist, anchor_x, anchor_y, anchor_z, anchor_id):
                skipped_counts['INVALID_RANGING'] += 1
                continue

            process_uwb(kf_data, anchor_x, anchor_y, anchor_z, dist, stddev, anchor_id)
            did_fuse = True

        elif mtype == 'POSITION':
            skipped_counts['POSITION'] += 1
            continue

        else:
            skipped_counts['UNKNOWN'] += 1
            continue

        if did_fuse:
            fused_count += 1
            if fused_count % log_interval == 0:
                state = get_state(kf_data)
                log['timestamp_ms'].append(ts)
                log['type'].append(mtype)
                log['imu_enabled'].append(bool(imu_enabled_current))
                for key in ['x', 'y', 'z', 'vx', 'vy', 'vz',
                            'roll', 'pitch', 'yaw', 'qw', 'qx', 'qy', 'qz']:
                    log[key].append(state[key])
                for i, name in enumerate(state_names):
                    log[f'P_{name}'].append(state['P_diag'][i])

        if (idx + 1) % 5000 == 0:
            print(f'  Processed {idx + 1}/{n_rows} rows...')

    results = pd.DataFrame(log)

    print(f'Done. {len(results)} state snapshots logged from {fused_count} fused measurements.')
    print(f'Filter horizon: {int(ordered_df["timestamp_ms"].iloc[0])} -> {int(ordered_df["timestamp_ms"].iloc[-1])} ms')
    print('IMU enable timeline used: no (disabled for this analysis)')
    print(f"Skipped POSITION rows: {skipped_counts['POSITION']}")
    print(f"Skipped disabled-IMU rows: {skipped_counts['IMU_DISABLED']}")
    print(f"Skipped invalid IMU rows: {skipped_counts['INVALID_IMU']}")
    print(f"Skipped invalid RANGING rows: {skipped_counts['INVALID_RANGING']}")
    print(f"Skipped unknown rows: {skipped_counts['UNKNOWN']}")
    return results



In [ ]:
# Run the filter on loaded data
if df is not None:
    results = run_filter(df)
    display(results.head())
else:
    print('No data loaded. Place CSV in analysis/data/ and re-run cell 4.')
    results = None

In [ ]:
# UWB-only replay: process only RANGING/UWB rows (no IMU prediction)
if df is None:
    print('No data loaded. Run CSV load cell first.')
else:
    def run_filter_uwb_only(df, custom_params=None, log_interval=1, start_time_ms=None, end_time_ms=None):
        """Run EKF using only UWB ranging updates (no IMU predict steps)."""
        if df is None or len(df) == 0:
            print('Input DataFrame is empty.')
            return pd.DataFrame()

        if 'timestamp_ms' not in df.columns or 'type' not in df.columns:
            raise ValueError("CSV must contain at least 'timestamp_ms' and 'type' columns.")

        ordered_df = df.sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

        if start_time_ms is None:
            t0 = float(pd.to_numeric(ordered_df['timestamp_ms'], errors='coerce').min())
            start_time_ms = int(t0 + 400000.0)  # start offline analysis after 400 s
        ordered_df = ordered_df[pd.to_numeric(ordered_df['timestamp_ms'], errors='coerce') >= float(start_time_ms)]
        if end_time_ms is not None:
            ordered_df = ordered_df[pd.to_numeric(ordered_df['timestamp_ms'], errors='coerce') <= float(end_time_ms)]
        ordered_df = ordered_df.dropna(subset=['timestamp_ms']).reset_index(drop=True)

        if len(ordered_df) == 0:
            print('No rows in selected time horizon after filtering.')
            return pd.DataFrame()

        init_ts = int(float(ordered_df['timestamp_ms'].iloc[0]))
        kf_data, kf_params = init_filter(custom_params, init_timestamp_ms=init_ts)

        log = {
            'timestamp_ms': [], 'type': [],
            'x': [], 'y': [], 'z': [],
            'vx': [], 'vy': [], 'vz': [],
            'roll': [], 'pitch': [], 'yaw': [],
            'qw': [], 'qx': [], 'qy': [], 'qz': [],
        }
        state_names = ['X', 'Y', 'Z', 'PX', 'PY', 'PZ', 'D0', 'D1', 'D2']
        for name in state_names:
            log[f'P_{name}'] = []

        n_rows = len(ordered_df)
        fused_count = 0
        skipped_counts = {'NON_RANGING': 0, 'INVALID_RANGING': 0}

        for idx, row in ordered_df.iterrows():
            ts = int(float(row['timestamp_ms']))
            mtype = str(row['type']).strip().upper()

            if mtype not in ('RANGING', 'UWB'):
                skipped_counts['NON_RANGING'] += 1
                continue

            dist = _get_numeric(row, ['dist_m', 'distance'])
            anchor_x = _get_numeric(row, ['anchor_x'])
            anchor_y = _get_numeric(row, ['anchor_y'])
            anchor_z = _get_numeric(row, ['anchor_z'])
            anchor_id = _get_int(row, ['anchor_addr', 'anchor_id'])
            stddev = _get_numeric(row, ['stddev'])
            if stddev is None:
                stddev = DEFAULT_UWB_STDDEV

            if None in (dist, anchor_x, anchor_y, anchor_z, anchor_id):
                skipped_counts['INVALID_RANGING'] += 1
                continue

            # No IMU prediction in this mode: only grow covariance on ranging timestamps
            lib.kalmanCoreAddProcessNoise(byref(kf_data), byref(kf_params), c_uint32(ts))
            process_uwb(kf_data, anchor_x, anchor_y, anchor_z, dist, stddev, anchor_id)

            fused_count += 1
            if fused_count % log_interval == 0:
                state = get_state(kf_data)
                log['timestamp_ms'].append(ts)
                log['type'].append(mtype)
                for key in ['x', 'y', 'z', 'vx', 'vy', 'vz',
                            'roll', 'pitch', 'yaw', 'qw', 'qx', 'qy', 'qz']:
                    log[key].append(state[key])
                for i, name in enumerate(state_names):
                    log[f'P_{name}'].append(state['P_diag'][i])

            if (idx + 1) % 5000 == 0:
                print(f'  Processed {idx + 1}/{n_rows} rows...')

        res = pd.DataFrame(log)
        print(f'UWB-only done. {len(res)} state snapshots logged from {fused_count} ranging updates.')
        print(f"Skipped non-ranging rows: {skipped_counts['NON_RANGING']}")
        print(f"Skipped invalid ranging rows: {skipped_counts['INVALID_RANGING']}")
        return res

    uwb_only_results = run_filter_uwb_only(df)

    if uwb_only_results is not None and len(uwb_only_results) > 1:
        fig, axes = plt.subplots(2, 1, figsize=(10, 10), sharex=False)

        t = (uwb_only_results['timestamp_ms'] - uwb_only_results['timestamp_ms'].iloc[0]) / 1000.0
        axes[0].plot(t, uwb_only_results['x'], label='x', linewidth=1.0)
        axes[0].plot(t, uwb_only_results['y'], label='y', linewidth=1.0)
        axes[0].set_ylabel('Position (m)')
        axes[0].set_title('UWB-only EKF Position vs Time')
        axes[0].grid(True, alpha=0.3)
        axes[0].legend()

        axes[1].plot(uwb_only_results['x'], uwb_only_results['y'], color='tab:blue', linewidth=1.2, label='UWB-only EKF')
        axes[1].plot(uwb_only_results['x'].iloc[0], uwb_only_results['y'].iloc[0], 'go', markersize=8, label='Start')
        axes[1].plot(uwb_only_results['x'].iloc[-1], uwb_only_results['y'].iloc[-1], 'rs', markersize=8, label='End')
        axes[1].set_xlabel('x (m)')
        axes[1].set_ylabel('y (m)')
        axes[1].set_title('UWB-only EKF XY Trajectory')
        axes[1].set_xlim(-2.0, 15.0)
        axes[1].set_ylim(-1.0, 6.0)
        axes[1].set_aspect('equal', adjustable='box')
        axes[1].grid(True, alpha=0.3)
        axes[1].legend(loc='best')

        plt.tight_layout()
        plt.show()
    else:
        print('Not enough UWB-only output to plot.')


## 6. Visualization

In [ ]:
import matplotlib.pyplot as plt

def plot_results(results):
    """Generate standard analysis plots from filter results."""
    if results is None or len(results) == 0:
        print('No results to plot.')
        return

    t = (results['timestamp_ms'] - results['timestamp_ms'].iloc[0]) / 1000.0  # seconds

    fig, axes = plt.subplots(4, 1, figsize=(14, 16), sharex=True)

    # --- Position ---
    ax = axes[0]
    ax.plot(t, results['x'], label='X', linewidth=0.8)
    ax.plot(t, results['y'], label='Y', linewidth=0.8)
    ax.plot(t, results['z'], label='Z', linewidth=0.8)
    ax.set_ylabel('Position (m)')
    ax.set_title('Position Estimate')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # --- Velocity ---
    ax = axes[1]
    ax.plot(t, results['vx'], label='Vx', linewidth=0.8)
    ax.plot(t, results['vy'], label='Vy', linewidth=0.8)
    ax.plot(t, results['vz'], label='Vz', linewidth=0.8)
    ax.set_ylabel('Velocity (m/s)')
    ax.set_title('Velocity Estimate (World Frame)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # --- Attitude ---
    ax = axes[2]
    RAD2DEG = 180.0 / np.pi
    ax.plot(t, results['roll'] * RAD2DEG, label='Roll', linewidth=0.8)
    ax.plot(t, results['pitch'] * RAD2DEG, label='Pitch', linewidth=0.8)
    ax.plot(t, results['yaw'] * RAD2DEG, label='Yaw', linewidth=0.8)
    ax.set_ylabel('Angle (deg)')
    ax.set_title('Attitude Estimate (Euler Angles)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    # --- Covariance (position) ---
    ax = axes[3]
    ax.semilogy(t, results['P_X'], label='P_X', linewidth=0.8)
    ax.semilogy(t, results['P_Y'], label='P_Y', linewidth=0.8)
    ax.semilogy(t, results['P_Z'], label='P_Z', linewidth=0.8)
    ax.set_ylabel('Variance (m²)')
    ax.set_xlabel('Time (s)')
    ax.set_title('Position Covariance (Diagonal)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # --- 2D Trajectory ---
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.plot(results['x'], results['y'], linewidth=0.8, alpha=0.8)
    ax.plot(results['x'].iloc[0], results['y'].iloc[0], 'go', markersize=10, label='Start')
    ax.plot(results['x'].iloc[-1], results['y'].iloc[-1], 'rs', markersize=10, label='End')
    ax.set_xlabel('X (m)')
    ax.set_ylabel('Y (m)')
    ax.set_title('2D Trajectory (XY Plane)')
    ax.set_aspect('equal')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


print('Plotting functions defined.')

In [ ]:
if results is not None:
    plot_results(results)

## 7. Covariance & Innovation Analysis

In [ ]:
def plot_covariance_all(results):
    """Plot all 9 covariance diagonal elements."""
    if results is None or len(results) == 0:
        print('No results to plot.')
        return

    t = (results['timestamp_ms'] - results['timestamp_ms'].iloc[0]) / 1000.0
    state_names = ['X', 'Y', 'Z', 'PX', 'PY', 'PZ', 'D0', 'D1', 'D2']
    labels = ['Pos X', 'Pos Y', 'Pos Z', 'Vel X', 'Vel Y', 'Vel Z',
              'Att Roll', 'Att Pitch', 'Att Yaw']

    fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

    # Position covariance
    for i in range(3):
        axes[0].semilogy(t, results[f'P_{state_names[i]}'], label=labels[i], linewidth=0.8)
    axes[0].set_ylabel('Variance')
    axes[0].set_title('Position Covariance')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Velocity covariance
    for i in range(3, 6):
        axes[1].semilogy(t, results[f'P_{state_names[i]}'], label=labels[i], linewidth=0.8)
    axes[1].set_ylabel('Variance')
    axes[1].set_title('Velocity Covariance')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    # Attitude covariance
    for i in range(6, 9):
        axes[2].semilogy(t, results[f'P_{state_names[i]}'], label=labels[i], linewidth=0.8)
    axes[2].set_ylabel('Variance')
    axes[2].set_xlabel('Time (s)')
    axes[2].set_title('Attitude Error Covariance')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


if results is not None:
    plot_covariance_all(results)

## 8. Parameter Tuning Experiments

Use this section to compare filter behavior with different parameter settings.

In [ ]:
def compare_params(df, param_sets, labels):
    """
    Run the filter with multiple parameter sets and overlay results.

    Args:
        df: Input DataFrame
        param_sets: List of dicts (None = defaults)
        labels: List of label strings
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    for params, label in zip(param_sets, labels):
        print(f'Running: {label}...')
        res = run_filter(df, custom_params=params, log_interval=1)
        t = (res['timestamp_ms'] - res['timestamp_ms'].iloc[0]) / 1000.0

        axes[0, 0].plot(t, res['x'], label=label, linewidth=0.8, alpha=0.8)
        axes[0, 1].plot(t, res['y'], label=label, linewidth=0.8, alpha=0.8)
        axes[1, 0].plot(res['x'], res['y'], label=label, linewidth=0.8, alpha=0.8)
        axes[1, 1].semilogy(t, res['P_X'], label=label, linewidth=0.8, alpha=0.8)

    axes[0, 0].set_title('X Position'); axes[0, 0].set_ylabel('m'); axes[0, 0].legend()
    axes[0, 1].set_title('Y Position'); axes[0, 1].set_ylabel('m'); axes[0, 1].legend()
    axes[1, 0].set_title('XY Trajectory'); axes[1, 0].set_aspect('equal'); axes[1, 0].legend()
    axes[1, 1].set_title('P_X Covariance'); axes[1, 1].set_ylabel('Variance'); axes[1, 1].legend()

    for ax in axes.flat:
        ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


# Example: compare default vs higher process noise
# Uncomment and modify to run:
# if df is not None:
#     compare_params(df,
#         param_sets=[
#             None,  # defaults
#             {'procNoiseAcc_xy': 1.0, 'procNoiseVel': 0.1},  # more IMU uncertainty
#             {'procNoiseAcc_xy': 0.1, 'procNoiseVel': 0.001},  # less IMU uncertainty
#         ],
#         labels=['Default', 'High Proc Noise', 'Low Proc Noise']
#     )

## 9. Export Results

In [ ]:
if results is not None:
    output_path = os.path.join(DATA_DIR, 'filter_output.csv')
    results.to_csv(output_path, index=False)
    print(f'Results exported to {output_path}')

## 10. Video Comparison: Onboard vs Offline Sensor Fusion

Use this after generating `results` to visually compare onboard `POSITION` estimates from CSV against the offline EKF output.


In [ ]:
# Live comparison playback (full shared horizon): onboard POSITION (CSV) vs offline EKF results
if df is None or results is None or len(results) == 0:
    print('Need both loaded CSV data (`df`) and offline filter output (`results`) before running this cell.')
else:
    from IPython.display import HTML, display
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.animation as animation

    HISTORY_SEC = 100.0
    PLAYBACK_SPEED = 10.0
    TARGET_FPS = 20.0
    MAX_RENDER_FRAMES = 1000

    pos_csv = df[df['type'].astype(str).str.upper() == 'POSITION'].copy() if 'type' in df.columns else pd.DataFrame()
    pos_csv = pos_csv.dropna(subset=['timestamp_ms', 'pos_x', 'pos_y']).sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

    ekf = results.dropna(subset=['timestamp_ms', 'x', 'y']).sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

    if len(pos_csv) < 2 or len(ekf) < 2:
        print('Need at least 2 samples in both onboard POSITION and offline EKF results.')
    else:
        t0_ms = float(df['timestamp_ms'].min())
        t_csv = (pos_csv['timestamp_ms'].to_numpy(dtype=float) - t0_ms) / 1000.0
        t_ekf = (ekf['timestamp_ms'].to_numpy(dtype=float) - t0_ms) / 1000.0

        t_start = max(float(np.min(t_csv)), float(np.min(t_ekf)))
        t_end = min(float(np.max(t_csv)), float(np.max(t_ekf)))

        csv_mask = (t_csv >= t_start) & (t_csv <= t_end)
        ekf_mask = (t_ekf >= t_start) & (t_ekf <= t_end)

        pos_csv_w = pos_csv[csv_mask].reset_index(drop=True)
        ekf_w = ekf[ekf_mask].reset_index(drop=True)
        t_csv_w = t_csv[csv_mask]
        t_ekf_w = t_ekf[ekf_mask]

        if (t_end <= t_start) or len(pos_csv_w) < 2 or len(ekf_w) < 2:
            print(f'No usable overlap. onboard={len(pos_csv_w)}, offline={len(ekf_w)}, range=({t_start:.2f}, {t_end:.2f})s')
        else:
            # Pair raw onboard/offline samples by nearest timestamp (no interpolation/smoothing)
            cmp = pd.merge_asof(
                pos_csv_w[['timestamp_ms', 'pos_x', 'pos_y']].sort_values('timestamp_ms'),
                ekf_w[['timestamp_ms', 'x', 'y']].sort_values('timestamp_ms'),
                on='timestamp_ms',
                direction='nearest',
            ).dropna(subset=['x', 'y'])

            if len(cmp) < 2:
                print('Not enough nearest-timestamp pairs for comparison.')
                plt.close('all')
                raise SystemExit

            t_frame = (cmp['timestamp_ms'].to_numpy(dtype=float) - t0_ms) / 1000.0
            x_csv = cmp['pos_x'].to_numpy(dtype=float)
            y_csv = cmp['pos_y'].to_numpy(dtype=float)
            x_ekf = cmp['x'].to_numpy(dtype=float)
            y_ekf = cmp['y'].to_numpy(dtype=float)

            # Residual diagnostics over full overlap
            dx = x_ekf - x_csv
            dy = y_ekf - y_csv
            err = np.sqrt(dx * dx + dy * dy)

            # Estimate constant XY bias (helps distinguish shape vs offset mismatch)
            off_x = float(np.median(dx))
            off_y = float(np.median(dy))
            err_bias_corrected = np.sqrt((dx - off_x) ** 2 + (dy - off_y) ** 2)

            print(
                f'Overlap {t_start:.2f} -> {t_end:.2f} s (duration {t_end - t_start:.2f} s), '
                f'onboard n={len(pos_csv_w)}, offline n={len(ekf_w)}'
            )
            print(
                f'RMS error={float(np.sqrt(np.mean(err**2))):.3f} m, '
                f'median={float(np.median(err)):.3f} m, '
                f'95%={float(np.percentile(err,95)):.3f} m'
            )
            print(
                f'Best-fit constant offset: dx={off_x:.3f} m, dy={off_y:.3f} m | '
                f'bias-corrected RMS={float(np.sqrt(np.mean(err_bias_corrected**2))):.3f} m'
            )

            # Quick static diagnostics before animation
            fig_diag, ax_diag = plt.subplots(1, 2, figsize=(13, 5))
            ax_diag[0].plot(x_csv, y_csv, color='#1f77b4', linewidth=1.5, label='Onboard POSITION (CSV)')
            ax_diag[0].plot(x_ekf, y_ekf, color='#d62728', linewidth=1.5, label='Offline EKF (analysis)')
            ax_diag[0].set_title('Full-overlap trajectory')
            ax_diag[0].set_xlabel('pos_x (m)')
            ax_diag[0].set_ylabel('pos_y (m)')
            ax_diag[0].set_aspect('equal', adjustable='box')
            ax_diag[0].set_xlim(-2.0, 15.0)
            ax_diag[0].set_ylim(-1.0, 6.0)
            ax_diag[0].grid(True, alpha=0.3)
            ax_diag[0].legend(loc='best')

            t_rel_full = t_frame - t_frame[0]
            ax_diag[1].plot(t_rel_full, err, color='black', linewidth=1.2, label='|offline - onboard|')
            ax_diag[1].plot(t_rel_full, err_bias_corrected, color='gray', linewidth=1.0, linestyle='--', label='bias-corrected')
            ax_diag[1].set_title('Position residual over time')
            ax_diag[1].set_xlabel('Time in overlap (s)')
            ax_diag[1].set_ylabel('Error (m)')
            ax_diag[1].grid(True, alpha=0.3)
            ax_diag[1].legend(loc='best')
            plt.tight_layout()
            plt.show()

            # Frame decimation for browser-safe animation payload
            desired_data_dt = PLAYBACK_SPEED / TARGET_FPS
            dt_data = np.diff(t_rel_full)
            median_dt = np.median(dt_data) if len(dt_data) > 0 else desired_data_dt
            stride = max(1, int(np.round(desired_data_dt / max(median_dt, 1e-6))))
            frame_idx = np.arange(0, len(t_rel_full), stride, dtype=int)
            if frame_idx[-1] != len(t_rel_full) - 1:
                frame_idx = np.append(frame_idx, len(t_rel_full) - 1)
            if len(frame_idx) > MAX_RENDER_FRAMES:
                stride2 = int(np.ceil(len(frame_idx) / MAX_RENDER_FRAMES))
                frame_idx = frame_idx[::stride2]
                if frame_idx[-1] != len(t_rel_full) - 1:
                    frame_idx = np.append(frame_idx, len(t_rel_full) - 1)

            all_x = np.concatenate([x_csv, x_ekf])
            all_y = np.concatenate([y_csv, y_ekf])
            x_min, x_max = np.min(all_x), np.max(all_x)
            y_min, y_max = np.min(all_y), np.max(all_y)
            cx, cy = 0.5 * (x_min + x_max), 0.5 * (y_min + y_max)
            span = max(x_max - x_min, y_max - y_min)
            if span == 0:
                span = 1.0
            pad = 0.05 * span
            half = 0.5 * span + pad

            fig, ax = plt.subplots(1, 1, figsize=(7, 7))
            ax.set_xlim(-2.0, 15.0)
            ax.set_ylim(-1.0, 6.0)
            ax.set_aspect('equal', adjustable='box')
            ax.set_xlabel('pos_x (m)')
            ax.set_ylabel('pos_y (m)')
            ax.set_title('Onboard vs Offline Fusion Playback (shared horizon)')
            ax.grid(True, alpha=0.3)

            onboard_line, = ax.plot([], [], color='#1f77b4', linewidth=2.2, label='Onboard POSITION (CSV)')
            offline_line, = ax.plot([], [], color='#d62728', linewidth=2.2, label='Offline EKF (analysis)')
            onboard_pt, = ax.plot([], [], 'o', color='#1f77b4', markersize=5)
            offline_pt, = ax.plot([], [], 'o', color='#d62728', markersize=5)
            time_text = ax.text(0.02, 0.98, '', transform=ax.transAxes, va='top')
            ax.legend(loc='lower right')

            interval_ms = int(1000.0 / TARGET_FPS)

            def init():
                onboard_line.set_data([], [])
                offline_line.set_data([], [])
                onboard_pt.set_data([], [])
                offline_pt.set_data([], [])
                time_text.set_text('t = 0.00 s')
                return onboard_line, offline_line, onboard_pt, offline_pt, time_text

            def update(frame_no):
                i = frame_idx[frame_no]
                t_now = t_rel_full[i]
                t_window_start = max(0.0, t_now - HISTORY_SEC)
                j0 = np.searchsorted(t_rel_full, t_window_start, side='left')

                onboard_line.set_data(x_csv[j0:i+1], y_csv[j0:i+1])
                offline_line.set_data(x_ekf[j0:i+1], y_ekf[j0:i+1])
                onboard_pt.set_data([x_csv[i]], [y_csv[i]])
                offline_pt.set_data([x_ekf[i]], [y_ekf[i]])
                time_text.set_text(
                    f't = {t_rel_full[i]:0.2f} s  |  trail = {min(HISTORY_SEC, t_rel_full[i]):0.2f} s  |  speed = {PLAYBACK_SPEED:.0f}x'
                )
                return onboard_line, offline_line, onboard_pt, offline_pt, time_text

            plt.rcParams['animation.embed_limit'] = 120
            anim = animation.FuncAnimation(
                fig,
                update,
                frames=len(frame_idx),
                init_func=init,
                interval=interval_ms,
                blit=False,
                repeat=False,
            )

            globals()['_compare_anim_ref'] = anim

            rendered = False
            try:
                display(HTML(anim.to_jshtml(default_mode='once')))
                rendered = True
            except Exception as e:
                print(f'JS animation render failed: {e}')

            if not rendered:
                try:
                    display(HTML(anim.to_html5_video()))
                    rendered = True
                except Exception as e:
                    print(f'HTML5 video render failed: {e}')

            if not rendered:
                print('Animation could not be rendered inline. Consider reducing MAX_RENDER_FRAMES or TARGET_FPS.')

            plt.close(fig)


In [ ]:
# Static comparison: offline (UWB+IMU) vs onboard POSITION for t > 400 s
if df is None or results is None or len(results) == 0:
    print('Need both loaded CSV data (`df`) and offline filter output (`results`).')
else:
    pos_csv = df[df['type'].astype(str).str.upper() == 'POSITION'].copy() if 'type' in df.columns else pd.DataFrame()
    pos_csv = pos_csv.dropna(subset=['timestamp_ms', 'pos_x', 'pos_y']).sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

    ekf = results.dropna(subset=['timestamp_ms', 'x', 'y']).sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

    if len(pos_csv) < 2 or len(ekf) < 2:
        print('Need at least 2 samples in both onboard and offline outputs.')
    else:
        t0 = float(df['timestamp_ms'].min())
        t_pos = (pos_csv['timestamp_ms'].to_numpy(dtype=float) - t0) / 1000.0
        t_ekf = (ekf['timestamp_ms'].to_numpy(dtype=float) - t0) / 1000.0

        pos_mask = t_pos > 400.0
        ekf_mask = t_ekf > 400.0

        pos_w = pos_csv[pos_mask].reset_index(drop=True)
        ekf_w = ekf[ekf_mask].reset_index(drop=True)

        if len(pos_w) < 2 or len(ekf_w) < 2:
            print(f'Not enough samples after t>400s filter (onboard={len(pos_w)}, offline={len(ekf_w)}).')
        else:
            fig, ax = plt.subplots(1, 1, figsize=(8, 6))
            ax.plot(pos_w['pos_x'].to_numpy(dtype=float), pos_w['pos_y'].to_numpy(dtype=float),
                    color='tab:blue', linewidth=1.5, label='Onboard POSITION (MCU)')
            ax.plot(ekf_w['x'].to_numpy(dtype=float), ekf_w['y'].to_numpy(dtype=float),
                    color='tab:red', linewidth=1.5, label='Offline EKF (UWB+IMU)')
            ax.set_title('Offline vs Onboard Position (t > 400 s)')
            ax.set_xlabel('x (m)')
            ax.set_ylabel('y (m)')
            ax.set_xlim(-2.0, 15.0)
            ax.set_ylim(-1.0, 6.0)
            ax.set_aspect('equal', adjustable='box')
            ax.grid(True, alpha=0.3)
            ax.legend(loc='best')
            plt.tight_layout()
            plt.show()


In [ ]:
# CEP50 analysis on onboard POSITION data for static 4x4 grid dwell windows
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

if df is None or 'type' not in df.columns:
    print('CSV dataframe `df` with a `type` column is required.')
else:
    pos = df[df['type'].astype(str).str.upper() == 'POSITION'].copy()
    pos = pos.dropna(subset=['timestamp_ms', 'pos_x', 'pos_y']).sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

    if len(pos) < 2:
        print('Not enough POSITION samples for CEP50 analysis.')
    else:
        # Exact user-provided dwell order and mapping (16 points)
        dwell_specs = [
            (9, 143, 0, 3),
            (224, 275, 0, 2),
            (303, 431, 0, 1),
            (486, 547, 0, 0),
            (584, 715, 1, 0),
            (758, 834, 1, 1),
            (857, 931, 1, 2),
            (955, 991, 1, 3),
            (1054, 1086, 2, 3),
            (1157, 1190, 2, 2),
            (1205, 1259, 2, 1),
            (1308, 1350, 2, 0),
            (1374, 1463, 3, 0),
            (1496, 1552, 3, 1),
            (1592, 1627, 3, 2),
            (1671, 1713, 3, 3),
        ]

        # Nominal grid base corner: given as y=0.570, x=5.145
        base_x = 5.145
        base_y = 0.570
        grid_spacing_m = 1.0

        # Time base anchored to first POSITION sample, matching provided intervals
        t0_ms = float(pos['timestamp_ms'].iloc[0])
        pos['t_sec'] = (pos['timestamp_ms'].to_numpy(dtype=float) - t0_ms) / 1000.0

        rows = []
        window_summary = []

        for t_start, t_end, gx, gy in dwell_specs:
            m = (pos['t_sec'] >= float(t_start)) & (pos['t_sec'] <= float(t_end))
            seg = pos[m].copy()

            gt_x = base_x + gx * grid_spacing_m
            gt_y = base_y + gy * grid_spacing_m

            if len(seg) > 0:
                seg['gx'] = int(gx)
                seg['gy'] = int(gy)
                seg['gt_x_nom'] = gt_x
                seg['gt_y_nom'] = gt_y
                seg['window_start_s'] = float(t_start)
                seg['window_end_s'] = float(t_end)
                rows.append(seg)

            window_summary.append({
                'window_s': f'{int(t_start)}-{int(t_end)}',
                'grid_pt': f'({int(gx)},{int(gy)})',
                'n_samples': int(len(seg)),
            })

        window_summary_df = pd.DataFrame(window_summary)
        print('Window coverage (exact requested order):')
        display(window_summary_df)

        if len(rows) == 0:
            print('No POSITION samples found inside the provided dwell windows.')
        else:
            eval_df = pd.concat(rows, ignore_index=True)
            est_x = eval_df['pos_x'].to_numpy(dtype=float)
            est_y = eval_df['pos_y'].to_numpy(dtype=float)
            gt_x_nom = eval_df['gt_x_nom'].to_numpy(dtype=float)
            gt_y_nom = eval_df['gt_y_nom'].to_numpy(dtype=float)

            # No-shift errors
            ex_nom = est_x - gt_x_nom
            ey_nom = est_y - gt_y_nom
            r_nom = np.sqrt(ex_nom * ex_nom + ey_nom * ey_nom)

            # Best global XY shift (least-squares)
            shift_x = float(np.mean(est_x - gt_x_nom))
            shift_y = float(np.mean(est_y - gt_y_nom))
            gt_x_shift = gt_x_nom + shift_x
            gt_y_shift = gt_y_nom + shift_y

            ex_shift = est_x - gt_x_shift
            ey_shift = est_y - gt_y_shift
            r_shift = np.sqrt(ex_shift * ex_shift + ey_shift * ey_shift)

            cep50_nom = float(np.percentile(r_nom, 50))
            cep50_shift = float(np.percentile(r_shift, 50))
            rmse_nom = float(np.sqrt(np.mean(r_nom * r_nom)))
            rmse_shift = float(np.sqrt(np.mean(r_shift * r_shift)))

            n_nonempty = int((window_summary_df['n_samples'] > 0).sum())
            print('Static-grid CEP50 summary (onboard POSITION):')
            print(f'  Samples used: {len(eval_df)} across {n_nonempty}/16 windows with data')
            print(f'  CEP50 (no shift):      {cep50_nom:.4f} m')
            print(f'  CEP50 (global shift):  {cep50_shift:.4f} m')
            print(f'  RMSE  (no shift):      {rmse_nom:.4f} m')
            print(f'  RMSE  (global shift):  {rmse_shift:.4f} m')
            print(f'  Best global shift: dx={shift_x:+.4f} m, dy={shift_y:+.4f} m')
            # Anchor map for this test (x, y, z)
            anchors = {
                'A3': (0.0, 0.09, 2.518),
                'A4': (0.0, 5.228, 1.286),
                'A5': (12.454, 5.773, 1.286),
                'A6': (12.454, 0.0, 2.38),
            }

            # Plot A: 2D full distribution with custom view window and corner-placed anchors
            x_min_plot, x_max_plot = 3.0, 10.0
            y_min_plot, y_max_plot = -0.1, 4.7

            fig, ax = plt.subplots(1, 1, figsize=(8, 6))
            ax.scatter(eval_df['pos_x'].to_numpy(dtype=float), eval_df['pos_y'].to_numpy(dtype=float),
                       s=10, color='tab:blue', alpha=0.6, label='State estimate')

            gt_pts = eval_df[['gx', 'gy', 'gt_x_nom', 'gt_y_nom']].drop_duplicates()
            gt_x_shift_pts = gt_pts['gt_x_nom'].to_numpy(dtype=float) + shift_x
            gt_y_shift_pts = gt_pts['gt_y_nom'].to_numpy(dtype=float) + shift_y
            ax.scatter(gt_x_shift_pts, gt_y_shift_pts, s=64, marker='x', color='tab:green', linewidths=2.0,
                       label='Ground truth')

            corner_positions = {
                'A3': (x_min_plot, y_min_plot),
                'A4': (x_min_plot, y_max_plot),
                'A5': (x_max_plot, y_max_plot),
                'A6': (x_max_plot, y_min_plot),
            }
            label_offsets = {
                'A3': (0.15, 0.10),
                'A4': (0.15, -0.20),
                'A5': (-2.25, -0.20),
                'A6': (-2.25, 0.10),
            }
            for name, (corner_x, corner_y) in corner_positions.items():
                ax.scatter([corner_x], [corner_y], s=120, marker='^', color='black', zorder=5)
                ox, oy = label_offsets[name]
                true_x, true_y, _ = anchors[name]
                ax.text(corner_x + ox, corner_y + oy, f'{name}: ({true_x:.3f}, {true_y:.3f})',
                        fontsize=13, fontweight='bold', color='black')

            ax.set_title('2D distribution of state estimates at static dwell locations', fontsize=16)
            ax.set_xlabel('x (m)', fontsize=14)
            ax.set_ylabel('y (m)', fontsize=14)
            ax.set_xlim(x_min_plot, x_max_plot)
            ax.set_ylim(y_min_plot, y_max_plot)
            ax.set_aspect('equal', adjustable='box')
            ax.tick_params(axis='both', labelsize=13)
            ax.grid(True, alpha=0.3)
            ax.legend(loc='best', fontsize=13)
            plt.tight_layout()
            export_path = 'cep50_distribution_static_dwells.png'
            fig.savefig(export_path, dpi=400, bbox_inches='tight')
            print(f'Saved high-resolution figure: {export_path}')
            plt.show()

            # Plot B: zoomed 2D view for first dwell (0,3) with CEP50 and 0.050 m circles
            first_t0, first_t1, first_gx, first_gy = dwell_specs[0]
            g03 = eval_df[(eval_df['window_start_s'] == float(first_t0)) &
                          (eval_df['window_end_s'] == float(first_t1)) &
                          (eval_df['gx'] == int(first_gx)) &
                          (eval_df['gy'] == int(first_gy))]

            if len(g03) > 0:
                g03_x = g03['pos_x'].to_numpy(dtype=float)
                g03_y = g03['pos_y'].to_numpy(dtype=float)
                c_x = base_x + first_gx * grid_spacing_m + shift_x
                c_y = base_y + first_gy * grid_spacing_m + shift_y

                r03 = np.sqrt((g03_x - c_x) ** 2 + (g03_y - c_y) ** 2)
                cep50_03 = float(np.percentile(r03, 50))

                fig, ax = plt.subplots(1, 1, figsize=(6, 6))
                ax.scatter(g03_x, g03_y, s=12, color='tab:blue', alpha=0.7, label='(0,3) dwell samples')
                ax.scatter([c_x], [c_y], s=40, marker='x', color='tab:green', label='Shifted dwell center')

                circle_cep = plt.Circle((c_x, c_y), cep50_03, color='tab:purple', fill=False, linewidth=2.0,
                                        label=f'CEP50 circle ({cep50_03:.4f} m)')
                circle_target = plt.Circle((c_x, c_y), 0.050, color='k', fill=False, linestyle='--', linewidth=1.5,
                                           label='0.050 m target circle')
                ax.add_patch(circle_cep)
                ax.add_patch(circle_target)

                pad = max(0.12, 1.2 * max(cep50_03, 0.050))
                ax.set_xlim(c_x - pad, c_x + pad)
                ax.set_ylim(c_y - pad, c_y + pad)
                ax.set_aspect('equal', adjustable='box')
                ax.set_title('Zoomed (0,3) dwell: samples and CEP circles')
                ax.set_xlabel('x (m)')
                ax.set_ylabel('y (m)')
                ax.grid(True, alpha=0.3)
                ax.legend(loc='best')
                plt.tight_layout()
                plt.show()
            else:
                print('No samples found for first dwell window (0,3), skipping zoomed CEP plot.')

            # Metric interpretation note
            print('Note: "CEP50 (global shift)" is pooled over all samples (sample-weighted).')
            print('      "Mean per-window CEP50" averages each dwell equally (window-weighted).')
            print('      For point-by-point grid performance, trust mean per-window CEP50 more.')

            # Per-window metrics in exact requested order (include missing windows)
            per_win = []
            for t_start, t_end, gx, gy in dwell_specs:
                g = eval_df[(eval_df['window_start_s'] == float(t_start)) &
                            (eval_df['window_end_s'] == float(t_end)) &
                            (eval_df['gx'] == int(gx)) &
                            (eval_df['gy'] == int(gy))]

                if len(g) == 0:
                    per_win.append({
                        'window_s': f'{int(t_start)}-{int(t_end)}',
                        'grid_pt': f'({int(gx)},{int(gy)})',
                        'n': 0,
                        'CEP50_no_shift_m': np.nan,
                        'CEP50_shifted_m': np.nan,
                        'RMSE_shifted_m': np.nan,
                    })
                    continue

                rr_nom = np.sqrt((g['pos_x'] - g['gt_x_nom'])**2 + (g['pos_y'] - g['gt_y_nom'])**2).to_numpy(dtype=float)
                rr_shift = np.sqrt((g['pos_x'] - (g['gt_x_nom'] + shift_x))**2 + (g['pos_y'] - (g['gt_y_nom'] + shift_y))**2).to_numpy(dtype=float)
                per_win.append({
                    'window_s': f'{int(t_start)}-{int(t_end)}',
                    'grid_pt': f'({int(gx)},{int(gy)})',
                    'n': int(len(g)),
                    'CEP50_no_shift_m': float(np.percentile(rr_nom, 50)),
                    'CEP50_shifted_m': float(np.percentile(rr_shift, 50)),
                    'RMSE_shifted_m': float(np.sqrt(np.mean(rr_shift**2))),
                })

            per_win_df = pd.DataFrame(per_win)
            mean_shifted_cep50 = float(np.nanmean(per_win_df['CEP50_shifted_m'].to_numpy(dtype=float)))
            print(f'Mean per-window CEP50 (global shift): {mean_shifted_cep50:.4f} m')
            display(per_win_df)

            # Time-series data over evaluation windows
            eval_df = eval_df.sort_values('t_sec', kind='stable').reset_index(drop=True)
            t = eval_df['t_sec'].to_numpy(dtype=float)

            e_shift = np.sqrt((eval_df['pos_x'].to_numpy(dtype=float) - (eval_df['gt_x_nom'].to_numpy(dtype=float) + shift_x))**2 +
                              (eval_df['pos_y'].to_numpy(dtype=float) - (eval_df['gt_y_nom'].to_numpy(dtype=float) + shift_y))**2)
            # Per-sample (non-accumulated) shifted XY error at each timestamp
            rmse_point = e_shift

            # Build shifted dwell reference points (piecewise-constant by window, no interpolation across bins)
            gt_t = []
            gt_x = []
            gt_y = []
            for t_start, t_end, gx, gy in dwell_specs:
                g = eval_df[(eval_df['window_start_s'] == float(t_start)) &
                            (eval_df['window_end_s'] == float(t_end)) &
                            (eval_df['gx'] == int(gx)) &
                            (eval_df['gy'] == int(gy))]
                if len(g) == 0:
                    continue
                tt = g['t_sec'].to_numpy(dtype=float)
                gt_t.append(tt)
                gt_x.append(np.full_like(tt, base_x + gx * grid_spacing_m + shift_x, dtype=float))
                gt_y.append(np.full_like(tt, base_y + gy * grid_spacing_m + shift_y, dtype=float))

            gt_t = np.concatenate(gt_t) if gt_t else np.array([], dtype=float)
            gt_x = np.concatenate(gt_x) if gt_x else np.array([], dtype=float)
            gt_y = np.concatenate(gt_y) if gt_y else np.array([], dtype=float)

            # Plot 1: x and y vs time (marker-only; no lines between bins)
            fig, ax = plt.subplots(1, 1, figsize=(12, 4))
            ax.scatter(t, eval_df['pos_x'].to_numpy(dtype=float), s=8, label='pos_x (onboard)', color='tab:blue', alpha=0.8)
            ax.scatter(t, eval_df['pos_y'].to_numpy(dtype=float), s=8, label='pos_y (onboard)', color='tab:orange', alpha=0.8)
            if len(gt_t) > 0:
                ax.scatter(gt_t, gt_x, s=4, marker='x', label='gt_x shifted (dwell)', color='navy', alpha=0.8)
                ax.scatter(gt_t, gt_y, s=4, marker='x', label='gt_y shifted (dwell)', color='darkorange', alpha=0.8)
            ax.set_title('Onboard POSITION and shifted dwell references vs time (evaluation windows)')
            ax.set_xlabel('Time (s)')
            ax.set_ylabel('Position (m)')
            ax.grid(True, alpha=0.3)
            ax.legend(loc='best', ncol=2)
            plt.tight_layout()
            plt.show()

            # Plot 2: per-sample shifted XY error vs time (no accumulation)
            fig, ax = plt.subplots(1, 1, figsize=(12, 4))
            ax.scatter(t, rmse_point, s=9, color='tab:red', label='Per-sample XY error (shifted GT)', alpha=0.85)
            ax.axhline(0.050, color='k', linestyle='--', linewidth=1.0, label='CEP50 target 0.050 m')
            ax.set_title('Per-sample shifted XY error vs time')
            ax.set_xlabel('Time (s)')
            ax.set_ylabel('Error (m)')
            ax.grid(True, alpha=0.3)
            ax.legend(loc='best')
            plt.tight_layout()
            plt.show()








In [ ]:
# Dynamic trajectory visualization with shifted rounded-rectangle ground truth
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize

if df is None or 'type' not in df.columns:
    print('CSV dataframe `df` with a `type` column is required.')
else:
    pos = df[df['type'].astype(str).str.upper() == 'POSITION'].copy()
    pos = pos.dropna(subset=['timestamp_ms', 'pos_x', 'pos_y']).sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

    if len(pos) < 2:
        print('Not enough POSITION samples for dynamic trajectory plot.')
    else:
        # --- Editable configuration ---
        T_START_SEC = 138.0
        T_END_SEC = 712.0
        SMOOTH_WINDOW = 199  # odd integer >=1; 1 disables smoothing

        X_MIN, X_MAX = 2.0, 10.0
        Y_MIN, Y_MAX = -1.0, 5.0

        # Ground-truth rounded rectangle geometry
        GT_WIDTH = 3.9   # meters
        GT_HEIGHT = 2.9  # meters
        GT_CORNER_RADIUS = 0.2  # meters

        # Initial center/rotation estimate before global-fit optimization
        GT_CENTER_X0 = 6.5
        GT_CENTER_Y0 = 2.3
        GT_ROT_DEG0 = 0.0     # initial rectangle yaw (deg)
        GT_ROT_OFFSET_DEG = 3.0  # extra constant CCW rotation applied after fitting
        GT_LINEWIDTH = 7.0    # ground-truth trace thickness (set smaller/larger as desired)

        # Anchors (actual coordinates), but rendered on plot corners
        anchors = {
            'A3': (0.0, 0.09, 2.518),
            'A4': (0.0, 5.228, 1.286),
            'A5': (12.454, 5.773, 1.286),
            'A6': (12.454, 0.0, 2.38),
        }

        # Time base anchored to first POSITION sample
        t0_ms = float(pos['timestamp_ms'].iloc[0])
        pos['t_sec'] = (pos['timestamp_ms'].to_numpy(dtype=float) - t0_ms) / 1000.0

        m = (pos['t_sec'] >= T_START_SEC) & (pos['t_sec'] <= T_END_SEC)
        seg = pos[m].copy().reset_index(drop=True)

        if len(seg) < 5:
            print(f'Not enough POSITION samples in [{T_START_SEC}, {T_END_SEC}] s window.')
        else:
            x = seg['pos_x'].to_numpy(dtype=float)
            y = seg['pos_y'].to_numpy(dtype=float)
            t = seg['t_sec'].to_numpy(dtype=float)
            t_rel = t - T_START_SEC
            t_span = max(1e-9, T_END_SEC - T_START_SEC)

            # Optional global low-pass smoothing on selected window
            if int(SMOOTH_WINDOW) < 1:
                raise ValueError('SMOOTH_WINDOW must be >= 1')
            smooth_w = int(SMOOTH_WINDOW)
            if smooth_w % 2 == 0:
                smooth_w += 1  # force odd window for symmetric smoothing

            if smooth_w > 1:
                x = pd.Series(x).rolling(window=smooth_w, center=True, min_periods=1).mean().to_numpy(dtype=float)
                y = pd.Series(y).rolling(window=smooth_w, center=True, min_periods=1).mean().to_numpy(dtype=float)
                print(f'Applied global smoothing: rolling mean window={smooth_w} samples')
            else:
                print('Smoothing disabled (SMOOTH_WINDOW=1)')

            # Build rounded-rectangle boundary centered at origin
            hw = GT_WIDTH / 2.0
            hh = GT_HEIGHT / 2.0
            r = GT_CORNER_RADIUS
            if r >= min(hw, hh):
                raise ValueError('GT_CORNER_RADIUS must be smaller than half-width/half-height.')

            n_line = 160
            n_arc = 100

            # Straight segments (excluding corner arcs)
            top_x = np.linspace(-hw + r, hw - r, n_line)
            top_y = np.full_like(top_x, hh)
            right_y = np.linspace(hh - r, -hh + r, n_line)
            right_x = np.full_like(right_y, hw)
            bot_x = np.linspace(hw - r, -hw + r, n_line)
            bot_y = np.full_like(bot_x, -hh)
            left_y = np.linspace(-hh + r, hh - r, n_line)
            left_x = np.full_like(left_y, -hw)

            # Corner arcs (clockwise)
            tr_th = np.linspace(np.pi/2, 0, n_arc)
            br_th = np.linspace(0, -np.pi/2, n_arc)
            bl_th = np.linspace(-np.pi/2, -np.pi, n_arc)
            tl_th = np.linspace(np.pi, np.pi/2, n_arc)

            tr_x, tr_y = (hw - r) + r*np.cos(tr_th), (hh - r) + r*np.sin(tr_th)
            br_x, br_y = (hw - r) + r*np.cos(br_th), (-hh + r) + r*np.sin(br_th)
            bl_x, bl_y = (-hw + r) + r*np.cos(bl_th), (-hh + r) + r*np.sin(bl_th)
            tl_x, tl_y = (-hw + r) + r*np.cos(tl_th), (hh - r) + r*np.sin(tl_th)

            bx0 = np.concatenate([top_x, tr_x, right_x, br_x, bot_x, bl_x, left_x, tl_x])
            by0 = np.concatenate([top_y, tr_y, right_y, br_y, bot_y, bl_y, left_y, tl_y])

            # Fast global shift + rotation fit using signed distance to a rounded rectangle
            # (optimize dx, dy, theta).
            def mse_for_params(dx, dy, theta):
                cx = GT_CENTER_X0 + dx
                cy = GT_CENTER_Y0 + dy
                ct = np.cos(theta)
                st = np.sin(theta)

                # Transform samples into rectangle-aligned frame via -theta rotation
                x_c = x - cx
                y_c = y - cy
                px =  ct * x_c + st * y_c
                py = -st * x_c + ct * y_c

                # SDF of rounded box (2D): abs distance to boundary is |sdf|
                qx = np.abs(px) - (hw - r)
                qy = np.abs(py) - (hh - r)
                outside = np.hypot(np.maximum(qx, 0.0), np.maximum(qy, 0.0))
                inside = np.minimum(np.maximum(qx, qy), 0.0)
                sdf = outside + inside - r
                return float(np.mean(sdf * sdf))

            # Pattern search (few evaluations, robust, no scipy dependency)
            best_dx, best_dy = 0.0, 0.0
            best_th = np.deg2rad(float(GT_ROT_DEG0))
            best_cost = mse_for_params(best_dx, best_dy, best_th)
            step_xy = 0.8
            step_th = np.deg2rad(6.0)

            for _ in range(14):
                improved = False
                for ddx in (0.0, step_xy, -step_xy):
                    for ddy in (0.0, step_xy, -step_xy):
                        for dth in (0.0, step_th, -step_th):
                            cx = best_dx + ddx
                            cy = best_dy + ddy
                            ct = best_th + dth
                            c = mse_for_params(cx, cy, ct)
                            if c < best_cost:
                                best_cost = c
                                best_dx, best_dy, best_th = float(cx), float(cy), float(ct)
                                improved = True

                if not improved:
                    step_xy *= 0.5
                    step_th *= 0.5

                if step_xy < 0.003 and step_th < np.deg2rad(0.05):
                    break

            gt_cx = GT_CENTER_X0 + best_dx
            gt_cy = GT_CENTER_Y0 + best_dy
            gt_th = best_th + np.deg2rad(float(GT_ROT_OFFSET_DEG))

            # Rotate boundary to world frame and shift to fitted center
            cth = np.cos(gt_th)
            sth = np.sin(gt_th)
            bx = gt_cx + cth * bx0 - sth * by0
            by = gt_cy + sth * bx0 + cth * by0

            # Time-colored scatter with fixed normalization to [T_START_SEC, T_END_SEC]
            norm = Normalize(vmin=0.0, vmax=t_span)
            cmap = cm.get_cmap('Spectral')

            fig, ax = plt.subplots(1, 1, figsize=(9, 7))
            sc = ax.scatter(x, y, c=t_rel, cmap=cmap, norm=norm, s=11, alpha=0.9, label='State estimate', zorder=4)

            ax.plot(bx, by, color='black', linewidth=GT_LINEWIDTH, label='Ground truth', zorder=2)

            # Put anchors at corners with actual (x,y) labels
            corner_positions = {
                'A3': (X_MIN, Y_MIN),
                'A4': (X_MIN, Y_MAX),
                'A5': (X_MAX, Y_MAX),
                'A6': (X_MAX, Y_MIN),
            }
            label_offsets = {
                'A3': (0.12, 0.10),
                'A4': (0.12, -0.24),
                'A5': (-2.20, -0.24),
                'A6': (-2.20, 0.10),
            }
            for name, (cx, cy) in corner_positions.items():
                ax.scatter([cx], [cy], s=120, marker='^', color='black', zorder=5)
                ox, oy = label_offsets[name]
                tx, ty, _ = anchors[name]
                ax.text(cx + ox, cy + oy, f'{name}: ({tx:.3f}, {ty:.3f})',
                        fontsize=13, fontweight='bold', color='black')

            cbar = fig.colorbar(sc, ax=ax, pad=0.02)
            cbar.set_label('Time (s)', fontsize=14)
            cbar.ax.tick_params(labelsize=12)

            ax.set_xlim(X_MIN, X_MAX)
            ax.set_ylim(Y_MIN, Y_MAX)
            ax.set_aspect('equal', adjustable='box')
            ax.set_xlabel('x (m)', fontsize=14)
            ax.set_ylabel('y (m)', fontsize=14)
            ax.tick_params(axis='both', labelsize=13)
            ax.grid(True, alpha=0.3)
            legend_y = Y_MIN + 1.18
            ax.legend(
                loc='upper center',
                bbox_to_anchor=((X_MIN + X_MAX) / 2.0, legend_y),
                bbox_transform=ax.transData,
                fontsize=12,
                ncol=2,
                frameon=True,
            )
            ax.set_title('State Estimate Under Dynamic Trajectory', fontsize=17)

            print(f'Fitted ground-truth center: ({gt_cx:.3f}, {gt_cy:.3f}) m')
            print(f'Applied shift from initial center ({GT_CENTER_X0:.3f}, {GT_CENTER_Y0:.3f}): '
                  f'dx={best_dx:+.3f} m, dy={best_dy:+.3f} m')
            print(f'Fitted ground-truth rotation (pre-offset): {np.rad2deg(best_th):+.3f} deg (initial {GT_ROT_DEG0:+.3f} deg)')
            print(f'Applied constant rotation offset: +{GT_ROT_OFFSET_DEG:.3f} deg (CCW) | final rotation={np.rad2deg(gt_th):+.3f} deg')

            plt.tight_layout()
            plt.show()

            # Final diagnostics: per-sample RMSE to nearest point on ground-truth boundary
            # (non-accumulated; each timestamp uses nearest boundary point).
            d2 = (x[:, None] - bx[None, :])**2 + (y[:, None] - by[None, :])**2
            rmse_t = np.sqrt(np.min(d2, axis=1))
            avg_rmse = float(np.sqrt(np.mean(rmse_t**2)))

            fig_err, ax_err = plt.subplots(1, 1, figsize=(9, 3.5))
            ax_err.scatter(t_rel, rmse_t, s=9, color='tab:red', alpha=0.85, label='Per-sample RMSE to nearest ground truth point')
            ax_err.axhline(avg_rmse, color='black', linestyle='--', linewidth=1.2, label=f'Average RMSE = {avg_rmse:.4f} m')
            ax_err.set_xlabel('Time (s)', fontsize=13)
            ax_err.set_ylabel('RMSE (m)', fontsize=13)
            ax_err.set_title('Per-sample RMSE vs time (nearest point on ground truth)', fontsize=14)
            ax_err.grid(True, alpha=0.3)
            ax_err.legend(loc='best', fontsize=10)
            plt.tight_layout()
            plt.show()

            print(f'Average RMSE (nearest point on ground-truth line): {avg_rmse:.4f} m')











In [ ]:
# IMU contribution probe (665s to 708s): init exactly at 665s, then IMU-only vs IMU+UWB
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ctypes

if df is None or 'type' not in df.columns:
    print('CSV dataframe `df` with a `type` column is required.')
else:
    T_START = 665.0
    T_END = 708.0
    YAW_VEC_DT = 2.0

    INIT_AT_PROBE = {
        'initialX': 4.4,
        'initialY': 1.1,
        'initialZ': 0.15,
        'initialYaw': np.deg2rad(270.0),
    }

    d = df.copy()
    d = d.dropna(subset=['timestamp_ms']).sort_values('timestamp_ms', kind='stable').reset_index(drop=True)

    pos_for_t0 = d[d['type'].astype(str).str.upper() == 'POSITION'] if 'type' in d.columns else pd.DataFrame()
    if len(pos_for_t0) > 0:
        t0_ms = float(pos_for_t0['timestamp_ms'].iloc[0])
        print(f'Probe time base: first POSITION timestamp ({int(t0_ms)} ms)')
    else:
        t0_ms = float(d['timestamp_ms'].iloc[0])
        print(f'Probe time base: first dataset timestamp ({int(t0_ms)} ms) [POSITION missing]')

    d['t_sec'] = (d['timestamp_ms'].to_numpy(dtype=float) - t0_ms) / 1000.0

    probe = d[(d['type'].astype(str).str.upper().isin(['IMU', 'RANGING', 'UWB', 'POSITION'])) & (d['t_sec'] >= T_START) & (d['t_sec'] <= T_END)].copy().reset_index(drop=True)

    if len(probe) == 0:
        print('No events in probe window.')
    else:
        probe_start_ts = int(float(probe['timestamp_ms'].iloc[0]))
        kf0, kf_params = init_filter(params=INIT_AT_PROBE, init_timestamp_ms=probe_start_ts)

        def clone_kf(src):
            dst = KalmanCoreData()
            ctypes.memmove(ctypes.byref(dst), ctypes.byref(src), ctypes.sizeof(KalmanCoreData))
            return dst

        def restore_kf(dst, src):
            ctypes.memmove(ctypes.byref(dst), ctypes.byref(src), ctypes.sizeof(KalmanCoreData))

        def finite_xyz(s):
            return np.isfinite(s['x']) and np.isfinite(s['y']) and np.isfinite(s['z'])

        def _num(row, keys):
            for k in keys:
                v = row.get(k, np.nan)
                if pd.notna(v):
                    try:
                        fv = float(v)
                    except Exception:
                        continue
                    if np.isfinite(fv):
                        return fv
            return None

        def _int_val(row, keys):
            v = _num(row, keys)
            if v is None:
                return None
            try:
                return int(v)
            except Exception:
                return None

        def replay_event_into(kf_data_obj, row, allow_uwb=True):
            et = str(row['type']).strip().upper()
            ts = int(float(row['timestamp_ms']))
            if et == 'IMU':
                ax = _num(row, ['accel_x']); ay = _num(row, ['accel_y']); az = _num(row, ['accel_z'])
                gx = _num(row, ['gyro_x']); gy = _num(row, ['gyro_y']); gz = _num(row, ['gyro_z'])
                if None in (ax, ay, az, gx, gy, gz):
                    return False
                process_imu(kf_data_obj, kf_params, ts, ax, ay, az, gx, gy, gz)
                return True
            if et in ('RANGING', 'UWB') and allow_uwb:
                dist = _num(row, ['dist_m', 'distance'])
                axx = _num(row, ['anchor_x']); axy = _num(row, ['anchor_y']); axz = _num(row, ['anchor_z'])
                aid = _int_val(row, ['anchor_addr', 'anchor_id'])
                stddev = _num(row, ['stddev'])
                if stddev is None:
                    stddev = float(DEFAULT_UWB_STDDEV) if 'DEFAULT_UWB_STDDEV' in globals() else 0.25
                if None in (dist, axx, axy, axz, aid) or (not np.isfinite(dist)) or dist <= 0.0:
                    return False
                process_uwb(kf_data_obj, axx, axy, axz, dist, stddev, aid)
                return True
            return False

        def apply_guarded(kf_data_obj, row, allow_uwb=True):
            prev = clone_kf(kf_data_obj)
            did = replay_event_into(kf_data_obj, row, allow_uwb=allow_uwb)
            if not did:
                return False, False
            s = get_state(kf_data_obj)
            if not finite_xyz(s):
                restore_kf(kf_data_obj, prev)
                return False, True
            return True, False

        kf_imu = clone_kf(kf0)
        kf_fused = clone_kf(kf0)
        s0 = get_state(kf0)
        print('Initialized exactly at probe start event:')
        print(f"  ts={probe_start_ts} ms, t={((probe_start_ts - t0_ms)/1000.0):.3f} s")
        print(f"  start state: x={s0['x']:.3f}, y={s0['y']:.3f}, z={s0['z']:.3f}, yaw={np.rad2deg(s0['yaw']):.2f} deg")

        probe_types = probe['type'].astype(str).str.upper().value_counts()
        print('Probe window raw events:', {k: int(v) for k, v in probe_types.items()})

        imu_hist, fused_hist = [], []
        diag = {'imu_events': 0, 'imu_updates_imu_only': 0, 'imu_updates_fused': 0, 'uwb_events': 0, 'uwb_updates_fused': 0, 'nan_rejects_imu_only': 0, 'nan_rejects_fused': 0}

        for row in probe.itertuples(index=False):
            mtype = str(row.type).strip().upper()
            if mtype == 'POSITION':
                continue
            if mtype == 'IMU':
                diag['imu_events'] += 1
            elif mtype in ('RANGING', 'UWB'):
                diag['uwb_events'] += 1

            rd = row._asdict()
            up_imu, rej_imu = apply_guarded(kf_imu, rd, allow_uwb=False)
            up_fus, rej_fus = apply_guarded(kf_fused, rd, allow_uwb=True)

            if rej_imu: diag['nan_rejects_imu_only'] += 1
            if rej_fus: diag['nan_rejects_fused'] += 1
            if mtype == 'IMU' and up_imu: diag['imu_updates_imu_only'] += 1
            if mtype == 'IMU' and up_fus: diag['imu_updates_fused'] += 1
            if mtype in ('RANGING', 'UWB') and up_fus: diag['uwb_updates_fused'] += 1

            t_sec = float(row.t_sec)
            if up_imu:
                s = get_state(kf_imu); imu_hist.append({'t': t_sec, 'x': s['x'], 'y': s['y'], 'yaw': s['yaw']})
            if up_fus:
                s = get_state(kf_fused); fused_hist.append({'t': t_sec, 'x': s['x'], 'y': s['y'], 'yaw': s['yaw']})

        print('Probe window effective updates:', diag)

        imu_df = pd.DataFrame(imu_hist)
        fus_df = pd.DataFrame(fused_hist)

        if len(imu_df) < 2 or len(fus_df) < 2:
            print(f'Not enough valid probe samples to plot (IMU-only={len(imu_df)}, IMU+UWB={len(fus_df)}).')
        else:
            imu_df = imu_df.sort_values('t', kind='stable').drop_duplicates('t').reset_index(drop=True)
            fus_df = fus_df.sort_values('t', kind='stable').drop_duplicates('t').reset_index(drop=True)
            imu_df['yaw_unwrap'] = np.unwrap(imu_df['yaw'].to_numpy(dtype=float))
            fus_df['yaw_unwrap'] = np.unwrap(fus_df['yaw'].to_numpy(dtype=float))

            fig, ax = plt.subplots(1, 1, figsize=(8, 6))
            ax.plot(imu_df['x'], imu_df['y'], color='tab:orange', linewidth=1.6, label='IMU-only state')
            ax.plot(fus_df['x'], fus_df['y'], color='tab:blue', linewidth=1.6, label='IMU+UWB state')
            t_vec = np.arange(T_START, T_END + 1e-9, YAW_VEC_DT)
            def draw_yaw_vectors(df_, color, scale=0.22):
                t_arr = df_['t'].to_numpy(dtype=float); x_arr = df_['x'].to_numpy(dtype=float)
                y_arr = df_['y'].to_numpy(dtype=float); yaw_arr = df_['yaw_unwrap'].to_numpy(dtype=float)
                if len(t_arr) < 2: return
                for tv in t_vec:
                    j = int(np.argmin(np.abs(t_arr - tv)))
                    vx = np.cos(yaw_arr[j]) * scale; vy = np.sin(yaw_arr[j]) * scale
                    ax.arrow(x_arr[j], y_arr[j], vx, vy, color=color, head_width=0.05, head_length=0.07, length_includes_head=True, alpha=0.9)
            draw_yaw_vectors(imu_df, 'tab:orange'); draw_yaw_vectors(fus_df, 'tab:blue')
            ax.set_title(f'IMU behavior probe: yaw vectors every {YAW_VEC_DT:.0f}s ({T_START:.0f}-{T_END:.0f}s)')
            ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)'); ax.grid(True, alpha=0.3); ax.legend(loc='best'); ax.set_aspect('equal', adjustable='box')
            plt.tight_layout(); plt.show()

            fig, ax = plt.subplots(1, 1, figsize=(9, 3.8))
            ax.plot(imu_df['t'] - T_START, np.rad2deg(imu_df['yaw_unwrap']), color='tab:orange', label='IMU-only yaw')
            ax.plot(fus_df['t'] - T_START, np.rad2deg(fus_df['yaw_unwrap']), color='tab:blue', label='IMU+UWB yaw')

            pos_ref = d[d['type'].astype(str).str.upper() == 'POSITION'].copy()
            yaw_col = None
            for c in ['yaw', 'pos_yaw', 'heading', 'heading_yaw']:
                if c in pos_ref.columns:
                    yaw_col = c; break
            if yaw_col is not None:
                pos_ref = pos_ref.dropna(subset=['timestamp_ms', yaw_col]).sort_values('timestamp_ms', kind='stable')
                pos_ref = pos_ref[(pos_ref['t_sec'] >= T_START) & (pos_ref['t_sec'] <= T_END)]
                if len(pos_ref) > 1:
                    yraw = pos_ref[yaw_col].to_numpy(dtype=float)
                    if np.nanpercentile(np.abs(yraw), 95) > (2.0 * np.pi + 0.5): yraw = np.deg2rad(yraw)
                    yref = np.unwrap(yraw)
                    ax.plot(pos_ref['t_sec'] - T_START, np.rad2deg(yref), color='tab:green', linewidth=1.2, label=f'Onboard POSITION yaw ({yaw_col})')
                else:
                    print('Onboard POSITION yaw exists, but insufficient samples in probe window.')
            else:
                print('No onboard yaw column found in POSITION rows (checked: yaw, pos_yaw, heading, heading_yaw).')

            ax.set_title('Yaw vs time in probe window')
            ax.set_xlabel('Time since 665s (s)'); ax.set_ylabel('Yaw (deg, unwrapped)')
            ax.grid(True, alpha=0.3); ax.legend(loc='best')
            plt.tight_layout(); plt.show()

